# Pipeline ETL — DataRetail 360 · fuentes heterogéneas, rechazos y enmascarado

Notebook único, ejecutable en **Google Colab** (`Entorno de ejecución → Ejecutar todo`).

Esta versión parte del modelo dimensional de `Tablas.md` (6 dimensiones + 4 hechos), pero cambia el enfoque de la capa Bronce:

- **Bronce heterogéneo:** imita CRM/API, ERP relacional, WMS/API, marketing, POS y MDM. No usa nombres `dim_*`/`fact_*` ni surrogate keys.
- **Idempotencia total:** la verificación ocurre antes de generar, extraer, transformar, cargar o enmascarar.
- **Rechazos separados por fuente:** un archivo por fuente, con especificación detallada y payload original.
- **Capa enmascarada:** `DataRetail360_Masked.db` con RUT y email protegidos.
- **Fechas de archivos:** `YYYYMMDD` para datos/rechazos; `YYYYMMDD_HHMMSS` solo para logs.

## Índice

| Paso | Sección |
|---|---|
| 0 | Dependencias y parámetros |
| 1 | Logging, idempotencia y fuentes |
| 2 | Generación Bronce heterogénea |
| 3 | EXTRACT, rechazo detallado y dimensiones |
| 4 | Transformación de hechos |
| 5 | LOAD, rechazos por fuente y enmascarado |
| 6 | Tests |
| 7 | Orquestación integral |

## 0. Dependencias y parámetros

Dependencias mínimas: `pandas` y `numpy`. En Colab ya vienen instaladas; la celda verifica el entorno.

In [ ]:
# Dependencias mínimas (Colab ya trae pandas/numpy)
import importlib.util
import subprocess
import sys

REQUIRED = {"pandas": "pandas", "numpy": "numpy"}
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Instalando:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencias listas ✔")

## 1. Parámetros, fuentes y logging

La corrida usa una fecha `YYYYMMDD` para nombrar datos y rechazos. El log usa `YYYYMMDD_HHMMSS`.
El encabezado `INICIO DE PIPELINE` se escribe como primera línea del log antes de cualquier paso E/T/L.

In [ ]:
"""ETL DataRetail 360 — fuentes heterogéneas, rechazos por fuente y capa enmascarada."""

# ============================================================
# 0. Dependencias y parámetros
# ============================================================
import importlib.util
import json
import logging
import re
import shutil
import sqlite3
import subprocess
import sys
import tempfile
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    display
except NameError:
    # display de Jupyter si existe; si no, imprime en consola.
    def display(obj):
        print(obj)

SEED = 42
N_VENTAS = 10_000_000
N_INVENTARIO = 120_000
N_CAMPANAS = 25_000
N_DEVOLUCIONES = 300_000
N_CLIENTES = 200_000
FECHA_INICIO = "2025-01-01"
FECHA_FIN = "2026-12-31"

DB_NAME = "DataRetail360.db"
MASKED_DB_NAME = "DataRetail360_Masked.db"

RUN_NOW = datetime.now()
RUN_DATE = RUN_NOW.strftime("%Y-%m-%d")
RUN_DATE_STR = RUN_NOW.strftime("%Y%m%d")
RUN_TIME = RUN_NOW.strftime("%Y%m%d_%H%M%S")
RUN_ID = RUN_TIME

DATA_DIR = Path("data")
LOG_DIR = Path("logs")
REJECTED_DIR = Path("rejected")

for _dir in (DATA_DIR, LOG_DIR, REJECTED_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(SEED)

MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
         "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

FUENTES = {
    "crm_clientes": {
        "sistema": "CRM REST API v2", "tipo": "API / JSON",
        "entidad": "clientes", "tabla_destino": "dim_cliente",
    },
    "erp_ventas": {
        "sistema": "ERP PostgreSQL", "tipo": "BD relacional (OLTP)",
        "entidad": "líneas de pedido", "tabla_destino": "fact_ventas",
    },
    "wms_inventario": {
        "sistema": "WMS REST API", "tipo": "API / JSON",
        "entidad": "snapshots de stock", "tabla_destino": "fact_inventario",
    },
    "mkt_campanas": {
        "sistema": "Marketing SaaS", "tipo": "export CSV",
        "entidad": "campañas", "tabla_destino": "fact_campanas",
    },
    "pos_devoluciones": {
        "sistema": "POS SQL Server", "tipo": "BD relacional",
        "entidad": "devoluciones", "tabla_destino": "fact_devoluciones",
    },
    "mdm_productos": {
        "sistema": "MDM Productos", "tipo": "BD relacional maestra",
        "entidad": "productos", "tabla_destino": "dim_producto",
    },
    "mdm_categorias": {
        "sistema": "MDM Productos", "tipo": "BD relacional maestra",
        "entidad": "categorías", "tabla_destino": "dim_categoria",
    },
    "mdm_sucursales": {
        "sistema": "MDM Sucursales", "tipo": "BD relacional maestra",
        "entidad": "sucursales", "tabla_destino": "dim_sucursal",
    },
    "mdm_canales": {
        "sistema": "MDM Canales", "tipo": "BD relacional maestra",
        "entidad": "canales", "tabla_destino": "dim_canal",
    },
    "core_calendario": {
        "sistema": "Data Platform", "tipo": "tabla de referencia",
        "entidad": "calendario", "tabla_destino": "dim_fecha",
    },
}

### Logging e idempotencia

La comprobación de idempotencia abre `DataRetail360.db` en **modo lectura** y consulta `etl_control`.
Si ya existe la fecha, el pipeline se detiene antes de crear cualquier archivo de datos.

In [ ]:
# ============================================================
# 1. Logging e idempotencia
# ============================================================
# Formato: timestamp | nivel | paso | mensaje.
class ETLFormatter(logging.Formatter):

    # Da formato a cada línea de log: timestamp | nivel | paso | mensaje.
    def format(self, record):
        step = getattr(record, "step", "-")
        return (
            f"{self.formatTime(record, '%Y-%m-%d %H:%M:%S')} | "
            f"{record.levelname:<8} | [{step}] | {record.getMessage()}"
        )


# Configura el logger 'etl': consola + archivo .log con timestamp completo.
def setup_logging(log_dir, run_time):
    log = logging.getLogger("etl")
    log.setLevel(logging.DEBUG)
    log.propagate = False
    for handler in list(log.handlers):
        log.removeHandler(handler)
        handler.close()

    formatter = ETLFormatter()
    console = logging.StreamHandler(sys.stdout)
    console.setFormatter(formatter)
    console.setLevel(logging.DEBUG)
    log.addHandler(console)

    log_file = Path(log_dir) / f"etl_{run_time}.log"
    file_handler = logging.FileHandler(log_file, mode="w", encoding="utf-8")
    file_handler.setFormatter(formatter)
    file_handler.setLevel(logging.DEBUG)
    log.addHandler(file_handler)
    return log, log_file


log, LOG_FILE = setup_logging(LOG_DIR, RUN_TIME)


# Escribe el encabezado de inicio como primeras líneas del log.
def registrar_inicio_log(run_id=RUN_ID, run_date=RUN_DATE, db_name=DB_NAME,
                         masked_db_name=MASKED_DB_NAME):
    log.info(
        "INICIO DE PIPELINE DataRetail 360 | run_id=%s | fecha=%s | db=%s | masked_db=%s",
        run_id, run_date, db_name, masked_db_name,
        extra={"step": "S"},
    )
    log.info(
        "Fuentes configuradas: %d | Bronce heterogéneo | rechazos separados por fuente",
        len(FUENTES), extra={"step": "S"},
    )
    log.debug(
        "Parámetros: seed=%s ventas=%s inventario=%s campañas=%s devoluciones=%s clientes=%s rango=%s..%s",
        SEED, N_VENTAS, N_INVENTARIO, N_CAMPANAS, N_DEVOLUCIONES, N_CLIENTES,
        FECHA_INICIO, FECHA_FIN, extra={"step": "S"},
    )


registrar_inicio_log()


# Abre SQLite en modo lectura; devuelve None si el archivo no existe.
def _abrir_sqlite_lectura(db_path):
    path = Path(db_path)
    if not path.exists():
        return None
    return sqlite3.connect(f"{path.resolve().as_uri()}?mode=ro", uri=True)


# Devuelve la fila de control del día o None. No crea ni modifica la base.
def ejecucion_previa(run_date=RUN_DATE, db_path=DB_NAME):
    con = _abrir_sqlite_lectura(db_path)
    if con is None:
        return None
    try:
        existe = con.execute(
            "SELECT 1 FROM sqlite_master WHERE type='table' AND name='etl_control'"
        ).fetchone()
        if not existe:
            return None
        row = con.execute(
            "SELECT run_id, ejecutado_en, filas_leidas, filas_ok, filas_rechazadas "
            "FROM etl_control WHERE run_date = ?",
            (run_date,),
        ).fetchone()
        if row is None:
            return None
        return {
            "run_id": row[0],
            "ejecutado_en": row[1],
            "filas_leidas": row[2],
            "filas_ok": row[3],
            "filas_rechazadas": row[4],
        }
    except sqlite3.Error:
        return None
    finally:
        con.close()

## 2. Generación Bronce heterogénea

Cada fuente conserva su propio esquema de origen:

| Fuente | Sistema simulado | Destino dimensional |
|---|---|---|
| `crm_clientes` | CRM REST API / JSON | `dim_cliente` |
| `erp_ventas` | ERP PostgreSQL / OLTP | `fact_ventas` |
| `wms_inventario` | WMS REST API / JSON | `fact_inventario` |
| `mkt_campanas` | Marketing SaaS / CSV | `fact_campanas` |
| `pos_devoluciones` | POS SQL Server | `fact_devoluciones` |
| `mdm_productos` | MDM / BD maestra | `dim_producto` |
| `mdm_categorias` | MDM / BD maestra | `dim_categoria` |
| `mdm_sucursales` | MDM / BD maestra | `dim_sucursal` |
| `mdm_canales` | MDM / BD maestra | `dim_canal` |
| `core_calendario` | Data Platform / referencia | `dim_fecha` |

Los RUT se generan con dígito verificador válido en al menos 90% de los casos. Cada CSV Bronce se guarda como
`data/{fuente}_{YYYYMMDD}.csv`.

El CRM **no expone una ID de cliente**: su llave primaria es el RUT. Por eso `crm_clientes` no tiene `customerId`
y las fuentes transaccionales referencian al cliente con `CUSTOMER_RUT` / `customer_rut`. En el datamart,
`rut_cliente` + `dv_cliente` son la clave natural y `cliente_key` es la ID de almacenamiento generada.

In [ ]:
# ============================================================
# 2. Generación de fuentes Bronce heterogéneas
# ============================================================
# Calcula el dígito verificador de un RUT chileno (módulo 11).
def dv_rut(n: int) -> str:
    s, m = 0, 2
    while n:
        s, n, m = s + (n % 10) * m, n // 10, 2 if m == 7 else m + 1
    r = 11 - s % 11
    return "0" if r == 11 else "K" if r == 10 else str(r)


# Normaliza un RUT de origen y devuelve (cuerpo, dv) o (None, None).
def parse_rut(raw):
    if raw is None or pd.isna(raw):
        return None, None
    limpio = re.sub(r"[^0-9Kk]", "", str(raw)).upper()
    if len(limpio) < 2:
        return None, None
    cuerpo, dv = limpio[:-1], limpio[-1]
    if not cuerpo.isdigit():
        return None, None
    return int(cuerpo), dv


# True si el RUT tiene cuerpo numérico y DV correcto.
def rut_valido(raw) -> bool:
    cuerpo, dv = parse_rut(raw)
    return cuerpo is not None and dv_rut(cuerpo) == dv


# Forma canónica body-DV usada para cruzar fuentes por RUT.
def _rut_canonico(raw):
    cuerpo, dv = parse_rut(raw)
    if cuerpo is None:
        return None
    return f"{cuerpo}-{dv}"


# Convierte nombres a una base simple para emails.
def _slug(texto):
    sin_acentos = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", ".", sin_acentos.lower()).strip(".")


# Fechas aleatorias dentro del rango del laboratorio.
def _fechas_iso(n, rng=RNG):
    inicio = pd.Timestamp(FECHA_INICIO)
    dias = (pd.Timestamp(FECHA_FIN) - inicio).days
    return (inicio + pd.to_timedelta(rng.integers(0, dias + 1, n), unit="D")).strftime("%Y-%m-%d")


NOMBRES = ["Ana", "Carlos", "María", "Pedro", "Lucía", "Jorge", "Carmen", "Luis", "Sofía", "Diego"]
APELLIDOS = ["Pérez", "González", "Rodríguez", "López", "Martínez", "García",
             "Sánchez", "Ramírez", "Torres", "Flores"]
REGIONES = ["Metropolitana", "Biobío", "Antofagasta", "Valparaíso", "Araucanía"]


# Fuente CRM/API: clientes con RUT formateado, email y campos de contacto.
def generar_crm_clientes(n=N_CLIENTES):
    rut_nums = RNG.choice(np.arange(6_000_000, 26_000_000), n, replace=False)
    dvs = [dv_rut(int(r)) for r in rut_nums]

    invalidos = RNG.choice(n, size=max(1, int(n * 0.09)), replace=False)
    for i in invalidos:
        dvs[i] = "0" if dvs[i] != "0" else "1"

    nombres = [NOMBRES[i % len(NOMBRES)] for i in range(n)]
    apellidos = [APELLIDOS[i % len(APELLIDOS)] for i in range(n)]
    emails = [f"{_slug(nombres[i])}.{_slug(apellidos[i])}.{i:06d}@example.cl" for i in range(n)]
    df = pd.DataFrame({
        "rut": [f"{int(r):,}".replace(",", ".") + "-" + d for r, d in zip(rut_nums, dvs)],
        "fullName": [f"{nombres[i]} {apellidos[i]}" for i in range(n)],
        "emailAddress": emails,
        "phone": [f"+569{RNG.integers(10_000_000, 99_999_999)}" for _ in range(n)],
        "regionName": RNG.choice(REGIONES, n),
        "status": RNG.choice(["ACTIVO", "INACTIVO"], n, p=[0.95, 0.05]),
        "lastUpdated": _fechas_iso(n),
    })

    inv_email = RNG.choice(n, size=max(1, int(n * 0.005)), replace=False)
    df.loc[inv_email, "emailAddress"] = "correo-invalido"

    nulos = RNG.choice(n, size=max(1, int(n * 0.002)), replace=False)
    df.loc[nulos, "rut"] = np.nan
    df.loc[RNG.choice(n, size=max(1, int(n * 0.001)), replace=False), "fullName"] = None

    duplicados = df.iloc[RNG.choice(n, size=max(1, int(n * 0.003)), replace=False)].copy()
    return pd.concat([df, duplicados], ignore_index=True)


# Fuente MDM: catálogo de categorías.
def generar_mdm_categorias():
    return pd.DataFrame({
        "category_code": ["C01", "C02", "C03", "C04", "C05"],
        "category_name": ["Tecnología", "Accesorios", "Gaming", "Electrohogar", "Hogar"],
        "description": ["Productos tecnológicos", "Accesorios y periféricos",
                        "Consolas y videojuegos", "Electrodomésticos", "Artículos para el hogar"],
    })


# Fuente MDM: productos maestros; incluye una fila duplicada y una inválida.
def generar_mdm_productos():
    nombres = [
        "Notebook 14", "Audífonos", "Smart TV 55", "Celular", "Tablet 10",
        "Monitor 27", "Teclado", "Mouse", "Impresora", "Parlante",
        "Smartwatch", "Cámara", "Router WiFi", "Disco SSD 1TB", "Consola",
        "Microondas", "Refrigerador", "Lavadora", "Aspiradora", "Ventilador",
    ]
    n = len(nombres)
    precios = RNG.integers(15_000, 800_000, n)
    costos = np.maximum(1, np.round(precios * RNG.uniform(0.40, 0.80, n))).astype(int)
    df = pd.DataFrame({
        "sku": [f"SKU-{i + 1:04d}" for i in range(n)],
        "product_name": nombres,
        "category_code": [f"C{(i % 5) + 1:02d}" for i in range(n)],
        "list_price": precios,
        "standard_cost": costos,
        "active": 1,
    })
    df = pd.concat([df, df.iloc[[0]].assign(product_name="Duplicado")], ignore_index=True)
    df = pd.concat([df, pd.DataFrame([{
        "sku": None, "product_name": "Producto sin SKU", "category_code": "C01",
        "list_price": -1, "standard_cost": 0, "active": 1,
    }])], ignore_index=True)
    return df


# Fuente MDM: sucursales.
def generar_mdm_sucursales():
    data = [
        ("S01", "Providencia", "Providencia", "Metropolitana", "09:00-21:00"),
        ("S02", "Las Condes", "Las Condes", "Metropolitana", "10:00-22:00"),
        ("S03", "Maipú", "Maipú", "Metropolitana", "09:00-21:00"),
        ("S04", "Ñuñoa", "Ñuñoa", "Metropolitana", "09:00-21:00"),
        ("S05", "Concepción", "Concepción", "Biobío", "09:00-21:00"),
        ("S06", "Antofagasta", "Antofagasta", "Antofagasta", "09:00-21:00"),
        ("S07", "Viña del Mar", "Viña del Mar", "Valparaíso", "10:00-22:00"),
        ("S08", "Temuco", "Temuco", "Araucanía", "09:00-21:00"),
    ]
    return pd.DataFrame(data, columns=["store_code", "store_name", "commune", "region", "opening_hours"])


# Fuente MDM: canales.
def generar_mdm_canales():
    return pd.DataFrame({
        "channel_code": ["POS", "WEB", "APP"],
        "channel_name": ["Tienda física", "E-commerce", "App"],
    })


# Fuente de referencia: calendario con feriados chilenos básicos.
def generar_core_calendario():
    fechas = pd.date_range(FECHA_INICIO, FECHA_FIN, freq="D")
    feriados = {"01-01": "Año Nuevo", "05-01": "Día del Trabajo",
                "09-18": "Independencia Nacional", "09-19": "Día de las Glorias del Ejército",
                "12-25": "Navidad"}
    claves = fechas.strftime("%m-%d")
    return pd.DataFrame({
        "calendar_date": fechas.strftime("%Y-%m-%d"),
        "is_holiday": [1 if k in feriados else 0 for k in claves],
        "holiday_name": [feriados.get(k) for k in claves],
    })

In [ ]:
# Índices aleatorios únicos para inyectar anomalías.
def _elegir(n, tasa, minimo=1):
    size = min(n, max(minimo, int(n * tasa)))
    return RNG.choice(n, size=size, replace=False)


# RUT válidos del CRM, que son su llave natural (el CRM no expone otra ID).
def _clientes_validos(clientes):
    return clientes.loc[clientes["rut"].map(rut_valido), "rut"].dropna().unique()


# Fuente ERP relacional: líneas de pedido con nombres de columnas OLTP.
def generar_erp_ventas(n=N_VENTAS, clientes=None, productos=None, sucursales=None, canales=None):
    clientes = clientes if clientes is not None else generar_crm_clientes()
    productos = productos if productos is not None else generar_mdm_productos()
    sucursales = sucursales if sucursales is not None else generar_mdm_sucursales()
    canales = canales if canales is not None else generar_mdm_canales()

    skus = productos["sku"].dropna().astype(str).str.upper().unique()
    precios = dict(zip(productos["sku"].astype(str).str.upper(), productos["list_price"]))
    stores = sucursales["store_code"].dropna().astype(str).unique()
    channels = canales["channel_code"].dropna().astype(str).unique()
    customers = _clientes_validos(clientes)
    fechas = _fechas_iso(n)

    sku_col = RNG.choice(skus, n)
    store_col = RNG.choice(stores, n)
    channel_col = RNG.choice(channels, n)
    customer_col = RNG.choice(customers, n)
    qty = RNG.integers(1, 5, n)
    precio = np.array([precios[s] for s in sku_col], dtype=float)
    descuento = np.where(RNG.random(n) < 0.20, np.round(precio * qty * 0.10, 2), 0.0)
    estado = RNG.choice(["COMPLETADA", "DEVUELTA", "ANULADA"], n, p=[0.90, 0.08, 0.02])

    df = pd.DataFrame({
        "ORDER_ID": [f"ORD-{i // 4 + 1:08d}" for i in range(n)],
        "LINE_NO": np.arange(n) % 4 + 1,
        "ORDER_TS": (pd.to_datetime(fechas) + pd.to_timedelta(RNG.integers(0, 86400, n), unit="s")).astype(str),
        "SKU": sku_col,
        "STORE_CODE": store_col,
        "CUSTOMER_RUT": customer_col,
        "CHANNEL_CODE": channel_col,
        "QTY": qty,
        "UNIT_PRICE": precio,
        "DISCOUNT_AMT": descuento,
        "TAX_AMT": np.round(precio * qty * 0.19, 2),
        "ORDER_STATUS": estado,
        "CURRENCY": "CLP",
    })

    idx = _elegir(n, 0.002)
    df.loc[idx, "QTY"] = 0
    df.loc[_elegir(n, 0.002), "UNIT_PRICE"] = -1
    df.loc[_elegir(n, 0.002), "ORDER_STATUS"] = "DESCONOCIDO"
    df.loc[_elegir(n, 0.002), "CUSTOMER_RUT"] = np.nan
    df.loc[_elegir(n, 0.001), "SKU"] = "SKU-NO-EXISTE"
    df.loc[_elegir(n, 0.001), "STORE_CODE"] = "S99"
    df.loc[_elegir(n, 0.001), "CHANNEL_CODE"] = "CANAL-X"
    df.loc[_elegir(n, 0.001), "DISCOUNT_AMT"] = 999_999_999
    df.loc[_elegir(n, 0.001), "ORDER_TS"] = "fecha-invalida"

    duplicados = df.iloc[_elegir(n, 0.002)].copy()
    return pd.concat([df, duplicados], ignore_index=True)


# Fuente WMS/API: snapshot de stock con fechas y códigos de almacén.
def generar_wms_inventario(n=N_INVENTARIO, productos=None, sucursales=None):
    productos = productos if productos is not None else generar_mdm_productos()
    sucursales = sucursales if sucursales is not None else generar_mdm_sucursales()
    skus = productos["sku"].dropna().astype(str).str.upper().unique()
    stores = sucursales["store_code"].dropna().astype(str).unique()
    fechas_unicas = pd.date_range(FECHA_INICIO, FECHA_FIN, freq="D").strftime("%Y-%m-%d")
    combinaciones = pd.MultiIndex.from_product(
        [fechas_unicas, skus, stores], names=["snapshotDate", "sku", "storeCode"]
    )
    elegidas = RNG.choice(len(combinaciones), size=min(n, len(combinaciones)), replace=False)
    base = pd.DataFrame(
        [combinaciones[i] for i in elegidas],
        columns=["snapshotDate", "sku", "storeCode"],
    )

    df = pd.DataFrame({
        "snapshotDate": base["snapshotDate"],
        "sku": base["sku"],
        "storeCode": base["storeCode"],
        "onHand": RNG.integers(0, 500, len(base)),
        "reorderPoint": RNG.integers(0, 100, len(base)),
        "lastCountAt": base["snapshotDate"],
        "warehouseCode": RNG.choice(["WH-01", "WH-02", "WH-03"], len(base)),
    })

    df.loc[_elegir(len(df), 0.002), "onHand"] = -5
    df.loc[_elegir(len(df), 0.002), "reorderPoint"] = -1
    df.loc[_elegir(len(df), 0.002), "sku"] = None
    df.loc[_elegir(len(df), 0.001), "storeCode"] = "S99"
    df.loc[_elegir(len(df), 0.001), "snapshotDate"] = "fecha-invalida"
    duplicados = df.iloc[_elegir(len(df), 0.002)].copy()
    return pd.concat([df, duplicados], ignore_index=True)


# Fuente Marketing SaaS: campañas con presupuesto e interacciones.
def generar_mkt_campanas(n=N_CAMPANAS, clientes=None, productos=None, canales=None):
    clientes = clientes if clientes is not None else generar_crm_clientes()
    productos = productos if productos is not None else generar_mdm_productos()
    canales = canales if canales is not None else generar_mdm_canales()
    skus = productos["sku"].dropna().astype(str).str.upper().unique()
    customers = _clientes_validos(clientes)
    channels = canales["channel_code"].dropna().astype(str).unique()

    inicio = pd.to_datetime(_fechas_iso(n))
    fin = (inicio + pd.to_timedelta(RNG.integers(7, 90, n), unit="D")).strftime("%Y-%m-%d")
    df = pd.DataFrame({
        "campaign_id": [f"CAMP-{i // 3 + 1:05d}" for i in range(n)],
        "campaign_name": [f"Campaña {i // 3 + 1}" for i in range(n)],
        "start_date": inicio.strftime("%Y-%m-%d"),
        "end_date": fin,
        "sku": RNG.choice(skus, n),
        "customer_rut": RNG.choice(customers, n),
        "channel_code": RNG.choice(channels, n),
        "budget_clp": RNG.integers(100_000, 5_000_000, n),
        "interactions": RNG.integers(10, 10_000, n),
        "status": RNG.choice(["ACTIVA", "PAUSADA"], n, p=[0.8, 0.2]),
    })

    df.loc[_elegir(n, 0.005), "end_date"] = "2020-01-01"
    df.loc[_elegir(n, 0.005), "budget_clp"] = -1
    df.loc[_elegir(n, 0.005), "customer_rut"] = np.nan
    df.loc[_elegir(n, 0.005), "sku"] = "SKU-NO-EXISTE"
    df.loc[_elegir(n, 0.005), "start_date"] = "fecha-invalida"
    duplicados = df.iloc[_elegir(n, 0.005)].copy()
    return pd.concat([df, duplicados], ignore_index=True)


# Fuente POS/SQL Server: devoluciones con ticket, motivo y reembolso.
def generar_pos_devoluciones(n=N_DEVOLUCIONES, clientes=None, productos=None, sucursales=None, canales=None):
    clientes = clientes if clientes is not None else generar_crm_clientes()
    productos = productos if productos is not None else generar_mdm_productos()
    sucursales = sucursales if sucursales is not None else generar_mdm_sucursales()
    canales = canales if canales is not None else generar_mdm_canales()
    skus = productos["sku"].dropna().astype(str).str.upper().unique()
    stores = sucursales["store_code"].dropna().astype(str).unique()
    channels = canales["channel_code"].dropna().astype(str).unique()
    customers = _clientes_validos(clientes)
    fechas = _fechas_iso(n)

    df = pd.DataFrame({
        "RETURN_ID": [f"RET-{i + 1:08d}" for i in range(n)],
        "ORDER_ID": [f"ORD-{RNG.integers(1, 50_000):08d}" for _ in range(n)],
        "RETURN_DATE": fechas,
        "SKU": RNG.choice(skus, n),
        "STORE_CODE": RNG.choice(stores, n),
        "CUSTOMER_RUT": RNG.choice(customers, n),
        "CHANNEL_CODE": RNG.choice(channels, n),
        "REASON": RNG.choice(["Producto defectuoso", "Talla incorrecta",
                              "No era lo esperado", "Llegó dañado"], n),
        "REFUND_AMT": RNG.integers(10_000, 500_000, n),
        "RETURN_STATUS": RNG.choice(["PROCESADA", "PENDIENTE"], n, p=[0.70, 0.30]),
    })

    df.loc[_elegir(n, 0.003), "REFUND_AMT"] = -1
    df.loc[_elegir(n, 0.003), "RETURN_STATUS"] = "DESCONOCIDO"
    df.loc[_elegir(n, 0.003), "CUSTOMER_RUT"] = np.nan
    df.loc[_elegir(n, 0.002), "RETURN_DATE"] = "fecha-invalida"
    duplicados = df.iloc[_elegir(n, 0.003)].copy()
    return pd.concat([df, duplicados], ignore_index=True)


# Persiste una fuente Bronce con timestamp de fecha YYYYMMDD.
def guardar_csv(df, fuente, run_date_str, data_dir=DATA_DIR):
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    path = data_dir / f"{fuente}_{run_date_str}.csv"
    df.to_csv(path, index=False)
    return path


# Genera las 10 fuentes heterogéneas y las persiste como CSV de fecha.
def generar_bronze(run_date_str=RUN_DATE_STR, data_dir=DATA_DIR):
    global RNG
    RNG = np.random.default_rng(SEED)
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    log.info("BRONZE inicio: generando %d fuentes heterogéneas", len(FUENTES), extra={"step": "E"})
    categorias = generar_mdm_categorias()
    productos = generar_mdm_productos()
    sucursales = generar_mdm_sucursales()
    canales = generar_mdm_canales()
    clientes = generar_crm_clientes()
    calendario = generar_core_calendario()
    ventas = generar_erp_ventas(clientes=clientes, productos=productos,
                                sucursales=sucursales, canales=canales)
    inventario = generar_wms_inventario(productos=productos, sucursales=sucursales)
    campanas = generar_mkt_campanas(clientes=clientes, productos=productos, canales=canales)
    devoluciones = generar_pos_devoluciones(clientes=clientes, productos=productos,
                                            sucursales=sucursales, canales=canales)

    fuentes = {
        "crm_clientes": clientes,
        "erp_ventas": ventas,
        "wms_inventario": inventario,
        "mkt_campanas": campanas,
        "pos_devoluciones": devoluciones,
        "mdm_productos": productos,
        "mdm_categorias": categorias,
        "mdm_sucursales": sucursales,
        "mdm_canales": canales,
        "core_calendario": calendario,
    }
    for nombre, df in fuentes.items():
        path = guardar_csv(df, nombre, run_date_str, data_dir)
        meta = FUENTES[nombre]
        log.info(
            "BRONZE %-16s sistema=%-20s formato=%-18s filas=%6d → %s",
            nombre, meta["sistema"], meta["tipo"], len(df), path.name,
            extra={"step": "E"},
        )
    log.info("BRONZE fin: %d fuentes persistidas", len(fuentes), extra={"step": "E"})
    return fuentes


# EXTRACT: lee CSV de fuentes; si falta alguno, lo regenera y persiste.
def extraer(run_date_str=RUN_DATE_STR, data_dir=DATA_DIR, fuentes=None, regenerar=None):
    data_dir = Path(data_dir)
    nombres = list(fuentes or FUENTES.keys())
    log.info("EXTRACT inicio: %d fuentes", len(nombres), extra={"step": "E"})
    faltantes = [n for n in nombres if not (data_dir / f"{n}_{run_date_str}.csv").exists()]
    if faltantes:
        log.warning("EXTRACT fuentes faltantes=%s; se regenera Bronce", faltantes, extra={"step": "E"})
        (regenerar or generar_bronze)(run_date_str, data_dir)

    datos = {}
    for nombre in nombres:
        path = data_dir / f"{nombre}_{run_date_str}.csv"
        df = pd.read_csv(path)
        datos[nombre] = df
        log.info("EXTRACT %-16s filas=%6d ← %s", nombre, len(df), path.name, extra={"step": "E"})
    log.info("EXTRACT fin", extra={"step": "E"})
    return datos

## 3. EXTRACT, rechazo detallado y dimensiones

Cada fila rechazada se escribe en `rejected/Rejected_{fuente}_{YYYYMMDD}.csv`, separada por fuente.
El detalle incluye: fuente, sistema de origen, tabla destino, fila, clave, campo, valor, motivo, regla,
severidad, acción sugerida y el payload original completo en JSON.

La razón `X`/`O` de la Bus Matrix se respeta en los hechos: `fact_inventario` no tiene canal ni cliente;
`fact_campanas` no tiene sucursal.

In [ ]:
# ============================================================
# 3. Framework de rechazo detallado y dimensiones
# ============================================================
MOTIVOS = {
    "CLAVE_NULA": {
        "descripcion": "Clave primaria o natural nula",
        "severidad": "ALTA",
        "accion": "Completar la clave en el sistema de origen y reenviar el registro",
    },
    "CLAVE_DUPLICADA": {
        "descripcion": "Clave natural duplicada (se conserva la primera ocurrencia)",
        "severidad": "MEDIA",
        "accion": "Deduplicar en origen y definir la versión vigente",
    },
    "RUT_INVALIDO": {
        "descripcion": "RUT con formato o dígito verificador inválido",
        "severidad": "ALTA",
        "accion": "Corregir el RUT en CRM y validar módulo 11",
    },
    "EMAIL_INVALIDO": {
        "descripcion": "Email con formato inválido",
        "severidad": "MEDIA",
        "accion": "Corregir el email de contacto o marcarlo como no disponible",
    },
    "NOMBRE_NULO": {
        "descripcion": "Nombre obligatorio nulo",
        "severidad": "MEDIA",
        "accion": "Completar el nombre del cliente en el sistema maestro",
    },
    "REGION_NULA": {
        "descripcion": "Región obligatoria nula",
        "severidad": "MEDIA",
        "accion": "Completar la región en el maestro de clientes",
    },
    "FECHA_INVALIDA": {
        "descripcion": "Fecha ausente o con formato inválido",
        "severidad": "ALTA",
        "accion": "Corregir la fecha en origen y reenviar",
    },
    "FECHA_FUERA_CALENDARIO": {
        "descripcion": "Fecha fuera del rango cubierto por dim_fecha",
        "severidad": "ALTA",
        "accion": "Revisar el rango del calendario o la fecha de origen",
    },
    "FK_INEXISTENTE": {
        "descripcion": "Clave foránea no encontrada en la dimensión correspondiente",
        "severidad": "ALTA",
        "accion": "Cargar la dimensión faltante o corregir la clave en origen",
    },
    "CANTIDAD_INVALIDA": {
        "descripcion": "Cantidad no numérica o no positiva",
        "severidad": "ALTA",
        "accion": "Corregir cantidad en el documento de origen",
    },
    "MONTO_NEGATIVO": {
        "descripcion": "Monto o reembolso negativo",
        "severidad": "ALTA",
        "accion": "Revisar signos y reversos en el sistema transaccional",
    },
    "DESCUENTO_INVALIDO": {
        "descripcion": "Descuento negativo o mayor al monto bruto",
        "severidad": "ALTA",
        "accion": "Validar la regla de descuento aplicada",
    },
    "ESTADO_DESCONOCIDO": {
        "descripcion": "Estado fuera del dominio permitido",
        "severidad": "MEDIA",
        "accion": "Mapear el estado de origen al dominio del datamart",
    },
    "FECHA_CAMPANA_INVALIDA": {
        "descripcion": "Fecha de término anterior a la fecha de inicio",
        "severidad": "MEDIA",
        "accion": "Corregir el rango de vigencia de la campaña",
    },
    "STOCK_NEGATIVO": {
        "descripcion": "Stock o punto de reposición negativo",
        "severidad": "ALTA",
        "accion": "Revisar el inventario físico y el snapshot del WMS",
    },
    "VALOR_NO_NUMERICO": {
        "descripcion": "Valor numérico obligatorio no convertible",
        "severidad": "ALTA",
        "accion": "Corregir el tipo de dato en la fuente",
    },
    "SKU_INVALIDO": {
        "descripcion": "SKU nulo o vacío",
        "severidad": "ALTA",
        "accion": "Completar el SKU en el maestro de productos",
    },
    "PRECIO_INVALIDO": {
        "descripcion": "Precio o costo nulo/negativo",
        "severidad": "ALTA",
        "accion": "Corregir el precio/costo en el maestro de productos",
    },
    "COSTO_MAYOR_PRECIO": {
        "descripcion": "Costo unitario mayor que el precio de venta",
        "severidad": "MEDIA",
        "accion": "Revisar la lista de precios y el costo estándar",
    },
    "DOMINIO_INVALIDO": {
        "descripcion": "Valor fuera del dominio permitido",
        "severidad": "MEDIA",
        "accion": "Corregir el valor en el sistema de origen",
    },
}

COLUMNAS_RECHAZO = [
    "rechazo_id", "fecha_proceso", "fuente", "sistema_origen", "tabla_destino",
    "fila_origen", "clave_registro", "campo_afectado", "valor_rechazado",
    "motivo_codigo", "motivo_descripcion", "regla", "severidad",
    "accion_sugerida", "payload_origen",
]


# Convierte valores numpy/pandas a tipos serializables en JSON.
def _serializable(valor):
    if valor is None:
        return None
    try:
        if pd.isna(valor):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(valor, (np.integer,)):
        return int(valor)
    if isinstance(valor, (np.floating,)):
        return float(valor)
    if isinstance(valor, (pd.Timestamp, datetime)):
        return valor.isoformat()
    return valor


# Serializa la fila original completa para auditoría manual.
def _payload_origen(fila):
    return json.dumps(
        {str(k): _serializable(v) for k, v in fila.items()},
        ensure_ascii=False,
        default=str,
    )


# Representación legible de la clave natural del registro.
def _clave_texto(fila, columnas):
    partes = []
    for col in columnas:
        valor = fila.get(col)
        partes.append("NULL" if _serializable(valor) is None else str(valor))
    return "|".join(partes)


# Agrega un rechazo detallado por cada fila que cumple la máscara.
def recolectar_rechazos(df, mascara, motivo, campo, fuente, regla,
                        clave_cols, run_date_str, rechazos):
    mascara = pd.Series(mascara, index=df.index).fillna(False).astype(bool)
    indices = set()
    meta = MOTIVOS[motivo]
    for idx in df.index[mascara]:
        fila = df.loc[idx]
        if campo and "|" in campo:
            valor = _clave_texto(fila, campo.split("|"))
        else:
            valor = fila.get(campo) if campo else None
        rechazos.append({
            "rechazo_id": f"RCH-{run_date_str}-{fuente[:4].upper()}-{len(rechazos) + 1:06d}",
            "fecha_proceso": run_date_str,
            "fuente": fuente,
            "sistema_origen": FUENTES[fuente]["sistema"],
            "tabla_destino": FUENTES[fuente]["tabla_destino"],
            "fila_origen": int(idx) + 2,
            "clave_registro": _clave_texto(fila, clave_cols),
            "campo_afectado": campo or "",
            "valor_rechazado": _serializable(valor),
            "motivo_codigo": motivo,
            "motivo_descripcion": meta["descripcion"],
            "regla": regla,
            "severidad": meta["severidad"],
            "accion_sugerida": meta["accion"],
            "payload_origen": _payload_origen(fila),
        })
        indices.add(idx)
    return indices


# Convierte una serie de fechas a fecha_key y devuelve máscaras de error.
def _fechas_a_key(series, fechas_validas):
    try:
        dt = pd.to_datetime(series, errors="coerce", format="mixed")
    except TypeError:
        dt = pd.to_datetime(series, errors="coerce")
    claves = pd.to_numeric(dt.dt.strftime("%Y%m%d"), errors="coerce").astype("Int64")
    invalidas = dt.isna()
    fuera = claves.notna() & ~claves.isin(fechas_validas)
    return claves, invalidas, fuera


# Construye dim_fecha a partir del calendario de referencia.
def construir_dim_fecha(calendario):
    fechas = pd.date_range(FECHA_INICIO, FECHA_FIN, freq="D")
    ref = calendario.copy()
    ref["calendar_date"] = pd.to_datetime(ref["calendar_date"], errors="coerce").dt.strftime("%Y-%m-%d")
    ref = ref.dropna(subset=["calendar_date"]).drop_duplicates("calendar_date")
    feriados = dict(zip(ref["calendar_date"], ref["is_holiday"]))
    return pd.DataFrame({
        "fecha_key": fechas.strftime("%Y%m%d").astype(int),
        "fecha": fechas.strftime("%Y-%m-%d"),
        "dia": fechas.day,
        "mes": fechas.month,
        "nombre_mes": [MESES[m - 1] for m in fechas.month],
        "trimestre": (fechas.month - 1) // 3 + 1,
        "anio": fechas.year,
        "dia_semana": fechas.dayofweek + 1,
        "nombre_dia": [DIAS[d] for d in fechas.dayofweek],
        "es_fin_semana": (fechas.dayofweek >= 5).astype(int),
        "es_feriado": [int(feriados.get(f, 0)) for f in fechas.strftime("%Y-%m-%d")],
    })


# Valida categorías y asigna surrogate keys.
def construir_dim_categoria(src, rechazos_fuente, run_date_str):
    df = src.copy()
    rechazados = set()
    codigo = df["category_code"].astype("string").str.strip().str.upper()
    clave = ["category_code"]

    m = df["category_code"].isna() | codigo.isna() | (codigo.str.len() == 0)
    rechazados |= recolectar_rechazos(df, m, "CLAVE_NULA", "category_code",
                                      "mdm_categorias", "category_code no nulo", clave,
                                      run_date_str, rechazos_fuente)
    m = df.duplicated(subset=["category_code"], keep="first")
    rechazados |= recolectar_rechazos(df, m, "CLAVE_DUPLICADA", "category_code",
                                      "mdm_categorias", "unicidad category_code", clave,
                                      run_date_str, rechazos_fuente)
    m = df["category_name"].isna() | (df["category_name"].astype("string").str.strip().str.len() == 0)
    rechazados |= recolectar_rechazos(df, m, "NOMBRE_NULO", "category_name",
                                      "mdm_categorias", "category_name no nulo", clave,
                                      run_date_str, rechazos_fuente)

    ok = df.loc[~df.index.isin(rechazados)].copy()
    ok["_code"] = ok["category_code"].astype("string").str.strip().str.upper()
    ok = ok.drop_duplicates("_code").reset_index(drop=True)
    dim = pd.DataFrame({
        "categoria_key": range(1, len(ok) + 1),
        "categoria": ok["category_name"].astype(str).str.strip(),
        "descripcion": ok["description"],
    })
    log.info("TRANSFORM mdm_categorias → dim_categoria filas_ok=%d rechazos=%d",
             len(dim), len(rechazos_fuente), extra={"step": "T"})
    return dim, dict(zip(ok["_code"], dim["categoria_key"]))


# Valida productos, normaliza SKU y resuelve la FK de categoría.
def construir_dim_producto(src, categoria_map, rechazos_fuente, run_date_str):
    df = src.copy()
    rechazados = set()
    sku = df["sku"].astype("string").str.strip().str.upper()
    clave = ["sku"]

    m = df["sku"].isna() | sku.isna() | (sku.str.len() == 0)
    rechazados |= recolectar_rechazos(df, m, "SKU_INVALIDO", "sku",
                                      "mdm_productos", "SKU no nulo", clave,
                                      run_date_str, rechazos_fuente)
    m = df.duplicated(subset=["sku"], keep="first")
    rechazados |= recolectar_rechazos(df, m, "CLAVE_DUPLICADA", "sku",
                                      "mdm_productos", "unicidad SKU", clave,
                                      run_date_str, rechazos_fuente)

    precio = pd.to_numeric(df["list_price"], errors="coerce")
    costo = pd.to_numeric(df["standard_cost"], errors="coerce")
    m = precio.isna() | (precio < 0)
    rechazados |= recolectar_rechazos(df, m, "PRECIO_INVALIDO", "list_price",
                                      "mdm_productos", "list_price >= 0", clave,
                                      run_date_str, rechazos_fuente)
    m = costo.isna() | (costo < 0)
    rechazados |= recolectar_rechazos(df, m, "PRECIO_INVALIDO", "standard_cost",
                                      "mdm_productos", "standard_cost >= 0", clave,
                                      run_date_str, rechazos_fuente)
    m = costo.notna() & precio.notna() & (costo > precio)
    rechazados |= recolectar_rechazos(df, m, "COSTO_MAYOR_PRECIO", "standard_cost",
                                      "mdm_productos", "costo <= precio", clave,
                                      run_date_str, rechazos_fuente)

    codigo_cat = df["category_code"].astype("string").str.strip().str.upper()
    cat_key = codigo_cat.map(categoria_map)
    m = cat_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "category_code",
                                      "mdm_productos", "categoría existente", clave,
                                      run_date_str, rechazos_fuente)

    ok = df.loc[~df.index.isin(rechazados)].copy()
    ok["_sku"] = ok["sku"].astype("string").str.strip().str.upper()
    ok["_cat"] = ok["category_code"].astype("string").str.strip().str.upper()
    ok = ok.drop_duplicates("_sku").reset_index(drop=True)
    dim = pd.DataFrame({
        "producto_key": range(1, len(ok) + 1),
        "sku": ok["_sku"],
        "producto": ok["product_name"].astype(str).str.strip(),
        "categoria_key": ok["_cat"].map(categoria_map).astype(int),
        "precio_venta": pd.to_numeric(ok["list_price"], errors="coerce").astype(float),
        "costo_unitario": pd.to_numeric(ok["standard_cost"], errors="coerce").astype(float),
    })
    log.info("TRANSFORM mdm_productos → dim_producto filas_ok=%d rechazos=%d",
             len(dim), len(rechazos_fuente), extra={"step": "T"})
    return dim


# Valida sucursales y asigna surrogate keys.
def construir_dim_sucursal(src, rechazos_fuente, run_date_str):
    df = src.copy()
    rechazados = set()
    clave = ["store_code"]
    m = df["store_code"].isna()
    rechazados |= recolectar_rechazos(df, m, "CLAVE_NULA", "store_code",
                                      "mdm_sucursales", "store_code no nulo", clave,
                                      run_date_str, rechazos_fuente)
    m = df.duplicated(subset=["store_code"], keep="first")
    rechazados |= recolectar_rechazos(df, m, "CLAVE_DUPLICADA", "store_code",
                                      "mdm_sucursales", "unicidad store_code", clave,
                                      run_date_str, rechazos_fuente)
    for col in ("store_name", "commune", "region"):
        m = df[col].isna()
        rechazados |= recolectar_rechazos(df, m, "NOMBRE_NULO", col,
                                          "mdm_sucursales", f"{col} no nulo", clave,
                                          run_date_str, rechazos_fuente)

    ok = df.loc[~df.index.isin(rechazados)].copy().drop_duplicates("store_code")
    ok = ok.reset_index(drop=True)
    dim = pd.DataFrame({
        "sucursal_key": range(1, len(ok) + 1),
        "sucursal": ok["store_name"].astype(str).str.strip(),
        "comuna": ok["commune"].astype(str).str.strip(),
        "region": ok["region"].astype(str).str.strip(),
        "horario_atencion": ok["opening_hours"],
    })
    log.info("TRANSFORM mdm_sucursales → dim_sucursal filas_ok=%d rechazos=%d",
             len(dim), len(rechazos_fuente), extra={"step": "T"})
    return dim, dict(zip(ok["store_code"], dim["sucursal_key"]))


# Valida canales y asigna surrogate keys.
def construir_dim_canal(src, rechazos_fuente, run_date_str):
    df = src.copy()
    rechazados = set()
    clave = ["channel_code"]
    m = df["channel_code"].isna()
    rechazados |= recolectar_rechazos(df, m, "CLAVE_NULA", "channel_code",
                                      "mdm_canales", "channel_code no nulo", clave,
                                      run_date_str, rechazos_fuente)
    m = df.duplicated(subset=["channel_code"], keep="first")
    rechazados |= recolectar_rechazos(df, m, "CLAVE_DUPLICADA", "channel_code",
                                      "mdm_canales", "unicidad channel_code", clave,
                                      run_date_str, rechazos_fuente)
    m = df["channel_name"].isna()
    rechazados |= recolectar_rechazos(df, m, "NOMBRE_NULO", "channel_name",
                                      "mdm_canales", "channel_name no nulo", clave,
                                      run_date_str, rechazos_fuente)

    ok = df.loc[~df.index.isin(rechazados)].copy().drop_duplicates("channel_code")
    ok = ok.reset_index(drop=True)
    dim = pd.DataFrame({
        "id_canal": range(1, len(ok) + 1),
        "descripcion": ok["channel_name"].astype(str).str.strip(),
    })
    log.info("TRANSFORM mdm_canales → dim_canal filas_ok=%d rechazos=%d",
             len(dim), len(rechazos_fuente), extra={"step": "T"})
    return dim, dict(zip(ok["channel_code"], dim["id_canal"]))


# Valida clientes y genera la clave de almacenamiento del datamart.
# El CRM no expone una ID de cliente: su llave primaria es el RUT. El datamart
# conserva ese RUT como clave natural y genera `cliente_key` como ID de almacenamiento.
def construir_dim_cliente(src, rechazos_fuente, run_date_str):
    df = src.copy()
    rechazados = set()
    clave = ["rut"]
    rut_par = [parse_rut(v) for v in df["rut"]]
    rut_num = [p[0] for p in rut_par]
    dv = [p[1] for p in rut_par]
    rut_canon = pd.Series([_rut_canonico(v) for v in df["rut"]], index=df.index)
    rut_ok = [cuerpo is not None and dv_rut(cuerpo) == digito
              for cuerpo, digito in zip(rut_num, dv)]

    m = df["rut"].isna()
    rechazados |= recolectar_rechazos(df, m, "CLAVE_NULA", "rut",
                                      "crm_clientes", "RUT no nulo", clave,
                                      run_date_str, rechazos_fuente)
    m = pd.Series([not ok for ok in rut_ok], index=df.index)
    rechazados |= recolectar_rechazos(df, m, "RUT_INVALIDO", "rut",
                                      "crm_clientes", "RUT módulo 11", clave,
                                      run_date_str, rechazos_fuente)
    email_ok = df["emailAddress"].astype("string").str.match(
        r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False)
    rechazados |= recolectar_rechazos(df, ~email_ok, "EMAIL_INVALIDO", "emailAddress",
                                      "crm_clientes", "email con formato válido", clave,
                                      run_date_str, rechazos_fuente)
    m = df["fullName"].isna() | (df["fullName"].astype("string").str.strip().str.len() == 0)
    rechazados |= recolectar_rechazos(df, m, "NOMBRE_NULO", "fullName",
                                      "crm_clientes", "fullName no nulo", clave,
                                      run_date_str, rechazos_fuente)
    m = df["regionName"].isna()
    rechazados |= recolectar_rechazos(df, m, "REGION_NULA", "regionName",
                                      "crm_clientes", "regionName no nulo", clave,
                                      run_date_str, rechazos_fuente)
    m = rut_canon.notna() & rut_canon.duplicated(keep="first")
    rechazados |= recolectar_rechazos(df, m, "CLAVE_DUPLICADA", "rut",
                                      "crm_clientes", "unicidad RUT", clave,
                                      run_date_str, rechazos_fuente)

    ok = df.loc[~df.index.isin(rechazados)].copy().reset_index(drop=True)
    ok["_rut_num"] = [parse_rut(v)[0] for v in ok["rut"]]
    ok["_dv"] = [parse_rut(v)[1] for v in ok["rut"]]
    ok["_rut_canon"] = [_rut_canonico(v) for v in ok["rut"]]
    ok["cliente_key"] = range(1, len(ok) + 1)
    valido_desde = pd.to_datetime(ok["lastUpdated"], errors="coerce").dt.strftime("%Y-%m-%d")
    valido_desde = valido_desde.fillna(run_date_str[:4] + "-" + run_date_str[4:6] + "-" + run_date_str[6:])
    dim = pd.DataFrame({
        "cliente_key": ok["cliente_key"],
        "rut_cliente": ok["_rut_num"].astype(int),
        "dv_cliente": ok["_dv"].astype(str),
        "nombre": ok["fullName"].astype(str).str.strip(),
        "region": ok["regionName"].astype(str).str.strip(),
        "valido_desde": valido_desde,
        "valido_hasta": None,
        "es_actual": 1,
    })
    emails = dict(zip(dim["cliente_key"], ok["emailAddress"].astype(str)))
    rut_map = dict(zip(ok["_rut_canon"], dim["cliente_key"]))
    log.info("TRANSFORM crm_clientes → dim_cliente filas_ok=%d rechazos=%d",
             len(dim), len(rechazos_fuente), extra={"step": "T"})
    return dim, emails, rut_map

## 4. Transformación de hechos

Las fuentes transaccionales se mapean a surrogate keys después de validar claves naturales, fechas,
dominios de estado, montos, cantidades y FKs. Las filas con uno o más problemas se derivan a su archivo
de rechazo de origen; no se cargan al datamart.

In [ ]:
# ============================================================
# 4. Transformación de hechos
# ============================================================
# Resuelve `cliente_key` desde el RUT que trae una fuente transaccional.
def _cliente_key_por_rut(series, cli_map):
    canon = series.map(_rut_canonico)
    return canon, canon.map(cli_map)


# Transforma líneas de pedido ERP a fact_ventas.
def construir_fact_ventas(src, dims, mapas, rechazos_fuente, run_date_str):
    df = src.copy()
    rechazados = set()
    clave = ["ORDER_ID", "LINE_NO"]
    fechas = set(dims["dim_fecha"]["fecha_key"].astype(int))
    fecha_key, fecha_mala, fecha_fuera = _fechas_a_key(df["ORDER_TS"], fechas)
    sku = df["SKU"].astype("string").str.strip().str.upper()
    prod_map = mapas["producto"]
    store_map = mapas["sucursal"]
    cli_map = mapas["cliente"]
    canal_map = mapas["canal"]

    m = df["ORDER_ID"].isna() | df["LINE_NO"].isna()
    rechazados |= recolectar_rechazos(df, m, "CLAVE_NULA", "ORDER_ID|LINE_NO",
                                      "erp_ventas", "clave de línea no nula", clave,
                                      run_date_str, rechazos_fuente)
    m = df.duplicated(subset=clave, keep="first")
    rechazados |= recolectar_rechazos(df, m, "CLAVE_DUPLICADA", "ORDER_ID|LINE_NO",
                                      "erp_ventas", "unicidad ORDER_ID+LINE_NO", clave,
                                      run_date_str, rechazos_fuente)
    rechazados |= recolectar_rechazos(df, fecha_mala, "FECHA_INVALIDA", "ORDER_TS",
                                      "erp_ventas", "ORDER_TS parseable", clave,
                                      run_date_str, rechazos_fuente)
    rechazados |= recolectar_rechazos(df, fecha_fuera, "FECHA_FUERA_CALENDARIO", "ORDER_TS",
                                      "erp_ventas", "fecha dentro de dim_fecha", clave,
                                      run_date_str, rechazos_fuente)

    qty = pd.to_numeric(df["QTY"], errors="coerce")
    m = qty.isna() | (qty <= 0)
    rechazados |= recolectar_rechazos(df, m, "CANTIDAD_INVALIDA", "QTY",
                                      "erp_ventas", "QTY > 0", clave,
                                      run_date_str, rechazos_fuente)
    precio = pd.to_numeric(df["UNIT_PRICE"], errors="coerce")
    m = precio.isna()
    rechazados |= recolectar_rechazos(df, m, "VALOR_NO_NUMERICO", "UNIT_PRICE",
                                      "erp_ventas", "UNIT_PRICE numérico", clave,
                                      run_date_str, rechazos_fuente)
    m = precio.notna() & (precio < 0)
    rechazados |= recolectar_rechazos(df, m, "MONTO_NEGATIVO", "UNIT_PRICE",
                                      "erp_ventas", "UNIT_PRICE >= 0", clave,
                                      run_date_str, rechazos_fuente)
    desc = pd.to_numeric(df["DISCOUNT_AMT"], errors="coerce")
    m = df["DISCOUNT_AMT"].notna() & desc.isna()
    rechazados |= recolectar_rechazos(df, m, "VALOR_NO_NUMERICO", "DISCOUNT_AMT",
                                      "erp_ventas", "DISCOUNT_AMT numérico", clave,
                                      run_date_str, rechazos_fuente)
    bruto = precio * qty
    m = desc.notna() & ((desc < 0) | (desc > bruto))
    rechazados |= recolectar_rechazos(df, m, "DESCUENTO_INVALIDO", "DISCOUNT_AMT",
                                      "erp_ventas", "0 <= descuento <= bruto", clave,
                                      run_date_str, rechazos_fuente)

    prod_key = sku.map(prod_map)
    m = df["SKU"].isna() | sku.isna() | prod_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "SKU",
                                      "erp_ventas", "SKU existe en dim_producto", clave,
                                      run_date_str, rechazos_fuente)
    store_key = df["STORE_CODE"].map(store_map)
    m = df["STORE_CODE"].isna() | store_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "STORE_CODE",
                                      "erp_ventas", "STORE_CODE existe en dim_sucursal", clave,
                                      run_date_str, rechazos_fuente)
    cli_canon, cli_key = _cliente_key_por_rut(df["CUSTOMER_RUT"], cli_map)
    m = df["CUSTOMER_RUT"].isna() | cli_canon.isna() | cli_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "CUSTOMER_RUT",
                                      "erp_ventas", "CUSTOMER_RUT existe en dim_cliente", clave,
                                      run_date_str, rechazos_fuente)
    canal_key = df["CHANNEL_CODE"].map(canal_map)
    m = df["CHANNEL_CODE"].isna() | canal_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "CHANNEL_CODE",
                                      "erp_ventas", "CHANNEL_CODE existe en dim_canal", clave,
                                      run_date_str, rechazos_fuente)

    estado_map = {"COMPLETADA": "OK", "DEVUELTA": "DEVUELTA", "ANULADA": "ANULADA"}
    estado = df["ORDER_STATUS"].map(estado_map)
    m = estado.isna()
    rechazados |= recolectar_rechazos(df, m, "ESTADO_DESCONOCIDO", "ORDER_STATUS",
                                      "erp_ventas", "estado mapeado a OK/DEVUELTA/ANULADA", clave,
                                      run_date_str, rechazos_fuente)

    ok = df.loc[~df.index.isin(rechazados)].copy()
    ok["_fecha_key"] = fecha_key.loc[ok.index]
    ok["_prod_key"] = prod_key.loc[ok.index]
    ok["_store_key"] = store_key.loc[ok.index]
    ok["_cli_key"] = cli_key.loc[ok.index]
    ok["_canal_key"] = canal_key.loc[ok.index]
    ok["_estado"] = estado.loc[ok.index]
    ok["_bruto"] = (precio * qty).loc[ok.index]
    ok["_desc"] = desc.loc[ok.index].fillna(0)
    ok = ok.reset_index(drop=True)
    fact = pd.DataFrame({
        "fact_key": range(1, len(ok) + 1),
        "fecha_key": ok["_fecha_key"].astype(int),
        "producto_key": ok["_prod_key"].astype(int),
        "sucursal_key": ok["_store_key"].astype(int),
        "cliente_key": ok["_cli_key"].astype(int),
        "id_canal": ok["_canal_key"].astype(int),
        "cantidad": pd.to_numeric(ok["QTY"], errors="coerce").astype(int),
        "monto_bruto": ok["_bruto"].round(2),
        "monto_neto": (ok["_bruto"] - ok["_desc"]).round(2),
        "estado": ok["_estado"],
        "monto_descuento": ok["_desc"].round(2),
    })
    log.info("TRANSFORM erp_ventas → fact_ventas filas_ok=%d rechazos=%d",
             len(fact), len(rechazos_fuente), extra={"step": "T"})
    return fact


# Transforma snapshots WMS a fact_inventario.
def construir_fact_inventario(src, dims, mapas, rechazos_fuente, run_date_str):
    df = src.copy()
    rechazados = set()
    clave = ["snapshotDate", "sku", "storeCode"]
    fechas = set(dims["dim_fecha"]["fecha_key"].astype(int))
    fecha_key, fecha_mala, fecha_fuera = _fechas_a_key(df["snapshotDate"], fechas)
    sku = df["sku"].astype("string").str.strip().str.upper()
    prod_map = mapas["producto"]
    store_map = mapas["sucursal"]

    m = df["snapshotDate"].isna() | df["sku"].isna() | df["storeCode"].isna()
    rechazados |= recolectar_rechazos(df, m, "CLAVE_NULA", "snapshotDate|sku|storeCode",
                                      "wms_inventario", "clave de snapshot no nula", clave,
                                      run_date_str, rechazos_fuente)
    m = df.duplicated(subset=clave, keep="first")
    rechazados |= recolectar_rechazos(df, m, "CLAVE_DUPLICADA", "snapshotDate|sku|storeCode",
                                      "wms_inventario", "unicidad snapshot+sku+store", clave,
                                      run_date_str, rechazos_fuente)
    rechazados |= recolectar_rechazos(df, fecha_mala, "FECHA_INVALIDA", "snapshotDate",
                                      "wms_inventario", "snapshotDate parseable", clave,
                                      run_date_str, rechazos_fuente)
    rechazados |= recolectar_rechazos(df, fecha_fuera, "FECHA_FUERA_CALENDARIO", "snapshotDate",
                                      "wms_inventario", "fecha dentro de dim_fecha", clave,
                                      run_date_str, rechazos_fuente)

    stock = pd.to_numeric(df["onHand"], errors="coerce")
    m = stock.isna() | (stock < 0)
    rechazados |= recolectar_rechazos(df, m, "STOCK_NEGATIVO", "onHand",
                                      "wms_inventario", "onHand >= 0", clave,
                                      run_date_str, rechazos_fuente)
    reposicion = pd.to_numeric(df["reorderPoint"], errors="coerce")
    m = df["reorderPoint"].notna() & reposicion.isna()
    rechazados |= recolectar_rechazos(df, m, "VALOR_NO_NUMERICO", "reorderPoint",
                                      "wms_inventario", "reorderPoint numérico", clave,
                                      run_date_str, rechazos_fuente)
    m = reposicion.notna() & (reposicion < 0)
    rechazados |= recolectar_rechazos(df, m, "STOCK_NEGATIVO", "reorderPoint",
                                      "wms_inventario", "reorderPoint >= 0", clave,
                                      run_date_str, rechazos_fuente)
    ultima_dt = pd.to_datetime(df["lastCountAt"], errors="coerce", format="mixed")
    m = ultima_dt.isna()
    rechazados |= recolectar_rechazos(df, m, "FECHA_INVALIDA", "lastCountAt",
                                      "wms_inventario", "lastCountAt parseable", clave,
                                      run_date_str, rechazos_fuente)

    prod_key = sku.map(prod_map)
    m = df["sku"].isna() | prod_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "sku",
                                      "wms_inventario", "SKU existe en dim_producto", clave,
                                      run_date_str, rechazos_fuente)
    store_key = df["storeCode"].map(store_map)
    m = df["storeCode"].isna() | store_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "storeCode",
                                      "wms_inventario", "storeCode existe en dim_sucursal", clave,
                                      run_date_str, rechazos_fuente)

    ok = df.loc[~df.index.isin(rechazados)].copy()
    ok["_fecha_key"] = fecha_key.loc[ok.index]
    ok["_prod_key"] = prod_key.loc[ok.index]
    ok["_store_key"] = store_key.loc[ok.index]
    ok["_stock"] = stock.loc[ok.index]
    ok["_reposicion"] = reposicion.loc[ok.index]
    ok["_ultima"] = ultima_dt.loc[ok.index].dt.strftime("%Y-%m-%d")
    ok = ok.reset_index(drop=True)
    fact = pd.DataFrame({
        "fact_key": range(1, len(ok) + 1),
        "fecha_key": ok["_fecha_key"].astype(int),
        "producto_key": ok["_prod_key"].astype(int),
        "sucursal_key": ok["_store_key"].astype(int),
        "cantidad_stock": ok["_stock"].astype(int),
        "stock_reposicion": ok["_reposicion"].astype("Int64"),
        "ultima_actualizacion": ok["_ultima"],
    })
    log.info("TRANSFORM wms_inventario → fact_inventario filas_ok=%d rechazos=%d",
             len(fact), len(rechazos_fuente), extra={"step": "T"})
    return fact


# Transforma campañas de marketing a fact_campanas.
def construir_fact_campanas(src, dims, mapas, rechazos_fuente, run_date_str):
    df = src.copy()
    rechazados = set()
    clave = ["campaign_id", "sku", "customer_rut", "channel_code"]
    fechas = set(dims["dim_fecha"]["fecha_key"].astype(int))
    fecha_key, fecha_mala, fecha_fuera = _fechas_a_key(df["start_date"], fechas)
    fin_dt = pd.to_datetime(df["end_date"], errors="coerce", format="mixed")
    inicio_dt = pd.to_datetime(df["start_date"], errors="coerce", format="mixed")
    sku = df["sku"].astype("string").str.strip().str.upper()
    prod_map = mapas["producto"]
    cli_map = mapas["cliente"]
    canal_map = mapas["canal"]

    m = df["campaign_id"].isna() | df["sku"].isna() | df["customer_rut"].isna() | df["channel_code"].isna()
    rechazados |= recolectar_rechazos(df, m, "CLAVE_NULA", "campaign_id|sku|customer_rut|channel_code",
                                      "mkt_campanas", "clave de campaña no nula", clave,
                                      run_date_str, rechazos_fuente)
    m = df.duplicated(subset=clave, keep="first")
    rechazados |= recolectar_rechazos(df, m, "CLAVE_DUPLICADA", "campaign_id|sku|customer_rut|channel_code",
                                      "mkt_campanas", "unicidad de campaña+producto+cliente+canal", clave,
                                      run_date_str, rechazos_fuente)
    rechazados |= recolectar_rechazos(df, fecha_mala, "FECHA_INVALIDA", "start_date",
                                      "mkt_campanas", "start_date parseable", clave,
                                      run_date_str, rechazos_fuente)
    rechazados |= recolectar_rechazos(df, fin_dt.isna(), "FECHA_INVALIDA", "end_date",
                                      "mkt_campanas", "end_date parseable", clave,
                                      run_date_str, rechazos_fuente)
    rechazados |= recolectar_rechazos(df, fecha_fuera, "FECHA_FUERA_CALENDARIO", "start_date",
                                      "mkt_campanas", "inicio dentro de dim_fecha", clave,
                                      run_date_str, rechazos_fuente)
    m = inicio_dt.notna() & fin_dt.notna() & (fin_dt < inicio_dt)
    rechazados |= recolectar_rechazos(df, m, "FECHA_CAMPANA_INVALIDA", "end_date",
                                      "mkt_campanas", "end_date >= start_date", clave,
                                      run_date_str, rechazos_fuente)

    inversion = pd.to_numeric(df["budget_clp"], errors="coerce")
    m = inversion.isna()
    rechazados |= recolectar_rechazos(df, m, "VALOR_NO_NUMERICO", "budget_clp",
                                      "mkt_campanas", "budget_clp numérico", clave,
                                      run_date_str, rechazos_fuente)
    m = inversion.notna() & (inversion < 0)
    rechazados |= recolectar_rechazos(df, m, "MONTO_NEGATIVO", "budget_clp",
                                      "mkt_campanas", "budget_clp >= 0", clave,
                                      run_date_str, rechazos_fuente)
    interacciones = pd.to_numeric(df["interactions"], errors="coerce")
    m = df["interactions"].notna() & interacciones.isna()
    rechazados |= recolectar_rechazos(df, m, "VALOR_NO_NUMERICO", "interactions",
                                      "mkt_campanas", "interactions numérico", clave,
                                      run_date_str, rechazos_fuente)
    m = interacciones.notna() & (interacciones < 0)
    rechazados |= recolectar_rechazos(df, m, "CANTIDAD_INVALIDA", "interactions",
                                      "mkt_campanas", "interactions >= 0", clave,
                                      run_date_str, rechazos_fuente)

    prod_key = sku.map(prod_map)
    m = df["sku"].isna() | prod_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "sku",
                                      "mkt_campanas", "SKU existe en dim_producto", clave,
                                      run_date_str, rechazos_fuente)
    cli_canon, cli_key = _cliente_key_por_rut(df["customer_rut"], cli_map)
    m = df["customer_rut"].isna() | cli_canon.isna() | cli_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "customer_rut",
                                      "mkt_campanas", "customer_rut existe en dim_cliente", clave,
                                      run_date_str, rechazos_fuente)
    canal_key = df["channel_code"].map(canal_map)
    m = df["channel_code"].isna() | canal_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "channel_code",
                                      "mkt_campanas", "channel_code existe en dim_canal", clave,
                                      run_date_str, rechazos_fuente)

    ok = df.loc[~df.index.isin(rechazados)].copy()
    ok["_fecha_key"] = fecha_key.loc[ok.index]
    ok["_prod_key"] = prod_key.loc[ok.index]
    ok["_cli_key"] = cli_key.loc[ok.index]
    ok["_canal_key"] = canal_key.loc[ok.index]
    ok["_inversion"] = inversion.loc[ok.index]
    ok["_interacciones"] = interacciones.loc[ok.index]
    ok = ok.reset_index(drop=True)
    fact = pd.DataFrame({
        "fact_key": range(1, len(ok) + 1),
        "fecha_key": ok["_fecha_key"].astype(int),
        "producto_key": ok["_prod_key"].astype(int),
        "id_canal": ok["_canal_key"].astype(int),
        "cliente_key": ok["_cli_key"].astype(int),
        "monto_inversion": ok["_inversion"].astype(float).round(2),
        "numero_interacciones": ok["_interacciones"].astype("Int64"),
        "fecha_inicio_campaña": pd.to_datetime(ok["start_date"]).dt.strftime("%Y-%m-%d"),
        "fecha_termino_campaña": pd.to_datetime(ok["end_date"]).dt.strftime("%Y-%m-%d"),
    })
    log.info("TRANSFORM mkt_campanas → fact_campanas filas_ok=%d rechazos=%d",
             len(fact), len(rechazos_fuente), extra={"step": "T"})
    return fact


# Transforma devoluciones POS a fact_devoluciones.
def construir_fact_devoluciones(src, dims, mapas, rechazos_fuente, run_date_str):
    df = src.copy()
    rechazados = set()
    clave = ["RETURN_ID"]
    fechas = set(dims["dim_fecha"]["fecha_key"].astype(int))
    fecha_key, fecha_mala, fecha_fuera = _fechas_a_key(df["RETURN_DATE"], fechas)
    sku = df["SKU"].astype("string").str.strip().str.upper()
    prod_map = mapas["producto"]
    store_map = mapas["sucursal"]
    cli_map = mapas["cliente"]
    canal_map = mapas["canal"]

    m = df["RETURN_ID"].isna()
    rechazados |= recolectar_rechazos(df, m, "CLAVE_NULA", "RETURN_ID",
                                      "pos_devoluciones", "RETURN_ID no nulo", clave,
                                      run_date_str, rechazos_fuente)
    m = df.duplicated(subset=clave, keep="first")
    rechazados |= recolectar_rechazos(df, m, "CLAVE_DUPLICADA", "RETURN_ID",
                                      "pos_devoluciones", "unicidad RETURN_ID", clave,
                                      run_date_str, rechazos_fuente)
    rechazados |= recolectar_rechazos(df, fecha_mala, "FECHA_INVALIDA", "RETURN_DATE",
                                      "pos_devoluciones", "RETURN_DATE parseable", clave,
                                      run_date_str, rechazos_fuente)
    rechazados |= recolectar_rechazos(df, fecha_fuera, "FECHA_FUERA_CALENDARIO", "RETURN_DATE",
                                      "pos_devoluciones", "fecha dentro de dim_fecha", clave,
                                      run_date_str, rechazos_fuente)

    monto = pd.to_numeric(df["REFUND_AMT"], errors="coerce")
    m = monto.isna()
    rechazados |= recolectar_rechazos(df, m, "VALOR_NO_NUMERICO", "REFUND_AMT",
                                      "pos_devoluciones", "REFUND_AMT numérico", clave,
                                      run_date_str, rechazos_fuente)
    m = monto.notna() & (monto < 0)
    rechazados |= recolectar_rechazos(df, m, "MONTO_NEGATIVO", "REFUND_AMT",
                                      "pos_devoluciones", "REFUND_AMT >= 0", clave,
                                      run_date_str, rechazos_fuente)
    estado = df["RETURN_STATUS"].where(df["RETURN_STATUS"].isin(["PROCESADA", "PENDIENTE"]))
    m = estado.isna()
    rechazados |= recolectar_rechazos(df, m, "ESTADO_DESCONOCIDO", "RETURN_STATUS",
                                      "pos_devoluciones", "RETURN_STATUS en dominio", clave,
                                      run_date_str, rechazos_fuente)

    prod_key = sku.map(prod_map)
    m = df["SKU"].isna() | prod_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "SKU",
                                      "pos_devoluciones", "SKU existe en dim_producto", clave,
                                      run_date_str, rechazos_fuente)
    store_key = df["STORE_CODE"].map(store_map)
    m = df["STORE_CODE"].isna() | store_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "STORE_CODE",
                                      "pos_devoluciones", "STORE_CODE existe en dim_sucursal", clave,
                                      run_date_str, rechazos_fuente)
    cli_canon, cli_key = _cliente_key_por_rut(df["CUSTOMER_RUT"], cli_map)
    m = df["CUSTOMER_RUT"].isna() | cli_canon.isna() | cli_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "CUSTOMER_RUT",
                                      "pos_devoluciones", "CUSTOMER_RUT existe en dim_cliente", clave,
                                      run_date_str, rechazos_fuente)
    canal_key = df["CHANNEL_CODE"].map(canal_map)
    m = df["CHANNEL_CODE"].isna() | canal_key.isna()
    rechazados |= recolectar_rechazos(df, m, "FK_INEXISTENTE", "CHANNEL_CODE",
                                      "pos_devoluciones", "CHANNEL_CODE existe en dim_canal", clave,
                                      run_date_str, rechazos_fuente)

    ok = df.loc[~df.index.isin(rechazados)].copy()
    ok["_fecha_key"] = fecha_key.loc[ok.index]
    ok["_prod_key"] = prod_key.loc[ok.index]
    ok["_store_key"] = store_key.loc[ok.index]
    ok["_cli_key"] = cli_key.loc[ok.index]
    ok["_canal_key"] = canal_key.loc[ok.index]
    ok["_monto"] = monto.loc[ok.index]
    ok["_estado"] = estado.loc[ok.index]
    ok = ok.reset_index(drop=True)
    fact = pd.DataFrame({
        "fact_key": range(1, len(ok) + 1),
        "fecha_key": ok["_fecha_key"].astype(int),
        "producto_key": ok["_prod_key"].astype(int),
        "sucursal_key": ok["_store_key"].astype(int),
        "cliente_key": ok["_cli_key"].astype(int),
        "id_canal": ok["_canal_key"].astype(int),
        "id_devolucion": range(1, len(ok) + 1),
        "motivo": ok["REASON"],
        "monto": ok["_monto"].astype(float).round(2),
        "estado": ok["_estado"],
    })
    log.info("TRANSFORM pos_devoluciones → fact_devoluciones filas_ok=%d rechazos=%d",
             len(fact), len(rechazos_fuente), extra={"step": "T"})
    return fact

## 5. LOAD, rechazos por fuente y capa enmascarada

La carga usa `INSERT`/append sobre tablas creadas con DDL y `PRAGMA foreign_keys = ON`.
No usa `to_sql(if_exists="replace")` porque eso eliminaría las restricciones de FK.

Después de la carga principal se crea `DataRetail360_Masked.db`:
`dim_cliente` conserva `cliente_key` (ID de almacenamiento) y el nombre, pero reemplaza RUT y email por valores enmascarados.
La tabla `masking_audit` deja evidencia de la técnica aplicada.

In [ ]:
# ============================================================
# 5. Carga, rechazos por fuente y capa enmascarada
# ============================================================
DDL_PRE = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS dim_fecha (
    fecha_key INTEGER PRIMARY KEY, fecha TEXT NOT NULL UNIQUE,
    dia INTEGER NOT NULL, mes INTEGER NOT NULL, nombre_mes TEXT NOT NULL,
    trimestre INTEGER NOT NULL, anio INTEGER NOT NULL, dia_semana INTEGER NOT NULL,
    nombre_dia TEXT NOT NULL, es_fin_semana INTEGER NOT NULL, es_feriado INTEGER NOT NULL
);
CREATE TABLE IF NOT EXISTS dim_categoria (
    categoria_key INTEGER PRIMARY KEY, categoria TEXT NOT NULL, descripcion TEXT
);
CREATE TABLE IF NOT EXISTS dim_producto (
    producto_key INTEGER PRIMARY KEY, sku TEXT NOT NULL UNIQUE, producto TEXT NOT NULL,
    categoria_key INTEGER NOT NULL REFERENCES dim_categoria(categoria_key),
    precio_venta REAL NOT NULL, costo_unitario REAL NOT NULL
);
CREATE TABLE IF NOT EXISTS dim_sucursal (
    sucursal_key INTEGER PRIMARY KEY, sucursal TEXT NOT NULL, comuna TEXT NOT NULL,
    region TEXT NOT NULL, horario_atencion TEXT
);
CREATE TABLE IF NOT EXISTS dim_canal (
    id_canal INTEGER PRIMARY KEY, descripcion TEXT NOT NULL
);
"""

DDL_DIM_CLIENTE = """
CREATE TABLE IF NOT EXISTS dim_cliente (
    cliente_key INTEGER PRIMARY KEY,
    rut_cliente INTEGER NOT NULL UNIQUE, dv_cliente TEXT NOT NULL, nombre TEXT NOT NULL,
    region TEXT NOT NULL, valido_desde TEXT NOT NULL, valido_hasta TEXT,
    es_actual INTEGER NOT NULL DEFAULT 1
);
"""

DDL_DIM_CLIENTE_MASKED = """
CREATE TABLE IF NOT EXISTS dim_cliente (
    cliente_key INTEGER PRIMARY KEY,
    rut_enmascarado TEXT NOT NULL, dv_enmascarado TEXT NOT NULL, nombre TEXT NOT NULL,
    email_enmascarado TEXT, region TEXT NOT NULL, valido_desde TEXT NOT NULL,
    valido_hasta TEXT, es_actual INTEGER NOT NULL DEFAULT 1
);
"""

DDL_POST = """
CREATE TABLE IF NOT EXISTS fact_ventas (
    fact_key INTEGER PRIMARY KEY,
    fecha_key INTEGER NOT NULL REFERENCES dim_fecha(fecha_key),
    producto_key INTEGER NOT NULL REFERENCES dim_producto(producto_key),
    sucursal_key INTEGER NOT NULL REFERENCES dim_sucursal(sucursal_key),
    cliente_key INTEGER NOT NULL REFERENCES dim_cliente(cliente_key),
    id_canal INTEGER NOT NULL REFERENCES dim_canal(id_canal),
    cantidad INTEGER NOT NULL, monto_bruto REAL NOT NULL, monto_neto REAL NOT NULL,
    estado TEXT NOT NULL, monto_descuento REAL DEFAULT 0
);
CREATE TABLE IF NOT EXISTS fact_inventario (
    fact_key INTEGER PRIMARY KEY,
    fecha_key INTEGER NOT NULL REFERENCES dim_fecha(fecha_key),
    producto_key INTEGER NOT NULL REFERENCES dim_producto(producto_key),
    sucursal_key INTEGER NOT NULL REFERENCES dim_sucursal(sucursal_key),
    cantidad_stock INTEGER NOT NULL, stock_reposicion INTEGER,
    ultima_actualizacion TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS fact_campanas (
    fact_key INTEGER PRIMARY KEY,
    fecha_key INTEGER NOT NULL REFERENCES dim_fecha(fecha_key),
    producto_key INTEGER NOT NULL REFERENCES dim_producto(producto_key),
    id_canal INTEGER NOT NULL REFERENCES dim_canal(id_canal),
    cliente_key INTEGER NOT NULL REFERENCES dim_cliente(cliente_key),
    monto_inversion REAL NOT NULL, numero_interacciones INTEGER,
    "fecha_inicio_campaña" TEXT NOT NULL, "fecha_termino_campaña" TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS fact_devoluciones (
    fact_key INTEGER PRIMARY KEY,
    fecha_key INTEGER NOT NULL REFERENCES dim_fecha(fecha_key),
    producto_key INTEGER NOT NULL REFERENCES dim_producto(producto_key),
    sucursal_key INTEGER NOT NULL REFERENCES dim_sucursal(sucursal_key),
    cliente_key INTEGER NOT NULL REFERENCES dim_cliente(cliente_key),
    id_canal INTEGER NOT NULL REFERENCES dim_canal(id_canal),
    id_devolucion INTEGER NOT NULL, motivo TEXT, monto REAL NOT NULL, estado TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS etl_control (
    run_date TEXT PRIMARY KEY, run_id TEXT NOT NULL, ejecutado_en TEXT NOT NULL,
    filas_leidas INTEGER, filas_ok INTEGER, filas_rechazadas INTEGER
);
"""

DDL = DDL_PRE + DDL_DIM_CLIENTE + DDL_POST
MASKED_DDL = DDL_PRE + DDL_DIM_CLIENTE_MASKED + DDL_POST + """
CREATE TABLE IF NOT EXISTS masking_audit (
    fecha_proceso TEXT NOT NULL, tabla TEXT NOT NULL, campo TEXT NOT NULL,
    tecnica TEXT NOT NULL, filas INTEGER NOT NULL
);
"""

ORDEN_DIMS = ["dim_fecha", "dim_categoria", "dim_producto", "dim_sucursal", "dim_cliente", "dim_canal"]
ORDEN_FACTS = ["fact_ventas", "fact_inventario", "fact_campanas", "fact_devoluciones"]


# Abre SQLite, activa FKs y aplica el DDL del datamart.
def conectar(db_path=DB_NAME, ddl=DDL):
    con = sqlite3.connect(str(db_path))
    con.execute("PRAGMA foreign_keys = ON")
    con.executescript(ddl)
    return con


# Borra hechos y dimensiones en orden seguro para recargas controladas.
def _limpiar_tablas(con):
    for tabla in reversed(ORDEN_FACTS + ORDEN_DIMS):
        con.execute(f'DELETE FROM "{tabla}"')


# Inserta dimensiones primero y luego hechos con if_exists='append'.
def _insertar_tablas(con, dims, facts):
    conteos = {}
    for nombre in ORDEN_DIMS:
        if nombre in dims:
            dims[nombre].to_sql(nombre, con, if_exists="append", index=False)
            conteos[nombre] = len(dims[nombre])
            log.info("LOAD %-16s %6d filas", nombre, len(dims[nombre]), extra={"step": "L"})
    for nombre in ORDEN_FACTS:
        if nombre in facts:
            facts[nombre].to_sql(nombre, con, if_exists="append", index=False)
            conteos[nombre] = len(facts[nombre])
            log.info("LOAD %-16s %6d filas", nombre, len(facts[nombre]), extra={"step": "L"})
    return conteos


# LOAD: dimensiones primero, luego hechos, preservando FKs y sin duplicar.
def cargar(dims, facts, db_path=DB_NAME):
    log.info("LOAD inicio", extra={"step": "L"})
    con = conectar(db_path)
    try:
        with con:
            _limpiar_tablas(con)
            conteos = _insertar_tablas(con, dims, facts)
            fk = con.execute("PRAGMA foreign_key_check").fetchall()
            if fk:
                raise RuntimeError(f"Violaciones de FK tras la carga: {fk[:5]}")
        log.info("LOAD fin", extra={"step": "L"})
        return conteos
    finally:
        con.close()


# Escribe un archivo de rechazos separado por fuente y lo registra en logs.
def guardar_rechazos_por_fuente(rechazos_por_fuente, run_date_str, rejected_dir=REJECTED_DIR):
    rejected_dir = Path(rejected_dir)
    rejected_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    for fuente in FUENTES:
        filas = rechazos_por_fuente.get(fuente, [])
        meta = FUENTES[fuente]
        if not filas:
            log.info(
                "RECHAZO fuente=%-16s sistema=%-20s filas=0 archivo=(sin archivo) motivos={}",
                fuente, meta["sistema"], extra={"step": "T"},
            )
            continue
        df = pd.DataFrame(filas, columns=COLUMNAS_RECHAZO)
        resumen = df["motivo_codigo"].value_counts().to_dict()
        path = rejected_dir / f"Rejected_{fuente}_{run_date_str}.csv"
        df.to_csv(path, index=False)
        paths[fuente] = path
        log.warning(
            "RECHAZO fuente=%-16s sistema=%-20s filas=%-5d archivo=%s motivos=%s",
            fuente, meta["sistema"], len(df), path.name, resumen,
            extra={"step": "T"},
        )
    log.info("RECHAZO fin: %d fuentes con archivo separado", len(paths), extra={"step": "T"})
    return paths


# Enmascara un RUT conservando solo dos dígitos iniciales.
def _mask_rut(rut):
    texto = re.sub(r"\D", "", str(rut))
    return f"{texto[:2]}.***.***-*" if len(texto) >= 2 else "***"


# Enmascara un email conservando la primera letra y el dominio.
def _mask_email(email):
    texto = str(email or "").strip()
    if "@" not in texto:
        return "***"
    local, dominio = texto.split("@", 1)
    return f"{local[:1]}***@{dominio}"


# Copia el modelo procesado a otra SQLite con RUT y email enmascarados.
def exportar_enmascarado(dims, facts, emails, run_date_str, db_path=MASKED_DB_NAME):
    log.info("MASK inicio → %s", db_path, extra={"step": "L"})
    cliente = dims["dim_cliente"].copy()
    cliente["rut_enmascarado"] = cliente["rut_cliente"].map(_mask_rut)
    cliente["dv_enmascarado"] = "***"
    cliente["email_enmascarado"] = cliente["cliente_key"].map(emails).map(_mask_email)
    cliente = cliente.drop(columns=["rut_cliente", "dv_cliente"])
    cliente = cliente[[
        "cliente_key", "rut_enmascarado", "dv_enmascarado", "nombre",
        "email_enmascarado", "region", "valido_desde", "valido_hasta", "es_actual",
    ]]
    dims_masked = dict(dims)
    dims_masked["dim_cliente"] = cliente

    con = conectar(db_path, MASKED_DDL)
    try:
        with con:
            for tabla in reversed(ORDEN_FACTS + ORDEN_DIMS):
                con.execute(f'DELETE FROM "{tabla}"')
            con.execute("DELETE FROM masking_audit")
            conteos = _insertar_tablas(con, dims_masked, facts)
            auditoria = pd.DataFrame([
                (run_date_str, "dim_cliente", "rut_cliente", "parcial: 2 dígitos + DV oculto", len(cliente)),
                (run_date_str, "dim_cliente", "emailAddress", "parcial: primera letra + dominio", len(cliente)),
            ], columns=["fecha_proceso", "tabla", "campo", "tecnica", "filas"])
            auditoria.to_sql("masking_audit", con, if_exists="append", index=False)
        log.info("MASK fin: %d dimensiones + %d hechos | auditoría=%d filas",
                 len(dims_masked), len(facts), len(auditoria), extra={"step": "L"})
        return conteos
    finally:
        con.close()


# Registra la corrida del día para bloquear re-ejecuciones completas.
def registrar_ejecucion(run_id, filas_leidas, filas_ok, filas_rechazadas,
                        run_date=RUN_DATE, db_path=DB_NAME):
    con = conectar(db_path)
    try:
        con.execute(
            "INSERT OR REPLACE INTO etl_control VALUES (?, ?, ?, ?, ?, ?)",
            (run_date, run_id, datetime.now(timezone.utc).isoformat(timespec="seconds"),
             filas_leidas, filas_ok, filas_rechazadas),
        )
        con.commit()
    finally:
        con.close()

## 6. Tests

Pruebas autocontenidas para verificar fuentes heterogéneas, RUT, timestamps, extracción, rechazos
separados y detallados, logs, carga con FKs, capa enmascarada e idempotencia previa total.

In [ ]:
# ============================================================
# 6. Tests autocontenidos
# ============================================================
# Assert mínimo y autocontenido.
def _assert(cond, msg):
    if not cond:
        raise AssertionError(msg)


# Bronce contiene fuentes de origen, no tablas dimensionales.
def test_bronze_heterogeneo():
    _assert(set(BRONZE) == set(FUENTES), "Bronce no coincide con las fuentes configuradas")
    _assert(not any(n.startswith(("dim_", "fact_")) for n in BRONZE), "Bronce no debe tener nombres dimensionales")
    _assert({"rut", "emailAddress", "fullName"} <= set(BRONZE["crm_clientes"].columns),
            "crm_clientes no conserva el esquema de CRM/API")
    _assert("customerId" not in BRONZE["crm_clientes"].columns,
            "El CRM no debe exponer una ID extra: su llave es el RUT")
    _assert({"ORDER_ID", "QTY", "UNIT_PRICE", "ORDER_STATUS", "CUSTOMER_RUT"} <= set(BRONZE["erp_ventas"].columns),
            "erp_ventas no conserva el esquema OLTP")
    _assert("cliente_key" not in BRONZE["crm_clientes"].columns, "Bronce no debe tener surrogate keys")
    _assert("monto_neto" not in BRONZE["erp_ventas"].columns, "Bronce no debe tener columnas de hechos")


# dim_cliente no hereda una ID del CRM: el RUT es clave natural y cliente_key es la ID de almacenamiento.
def test_cliente_sin_id_origen():
    columnas = list(DIMS_OK["dim_cliente"].columns)
    _assert("cliente_id" not in columnas, "dim_cliente no debe conservar una ID de origen inexistente")
    _assert({"cliente_key", "rut_cliente", "dv_cliente"} <= set(columnas),
            "dim_cliente debe tener cliente_key y el RUT como clave natural")
    _assert(DIMS_OK["dim_cliente"]["cliente_key"].is_unique, "cliente_key no es única")
    _assert(DIMS_OK["dim_cliente"]["rut_cliente"].is_unique, "rut_cliente no es única")


# Al menos 90% de los RUT de la fuente CRM son válidos.
def test_rut_valido_90():
    df = BRONZE["crm_clientes"]
    validos = sum(rut_valido(r) for r in df["rut"])
    _assert(validos / len(df) >= 0.90, f"Solo {validos/len(df):.1%} RUT válidos")


# Los archivos no-log usan YYYYMMDD; los logs sí pueden usar hora.
def test_timestamps_solo_fecha():
    for path in list(DATA_DIR.glob(f"*_{RUN_DATE_STR}.csv")):
        _assert(re.fullmatch(r".+_\d{8}\.csv", path.name), f"Timestamp inválido en {path.name}")
    for path in REJECTED_PATHS.values():
        _assert(re.fullmatch(r"Rejected_.+_\d{8}\.csv", path.name), f"Timestamp inválido en {path.name}")
    _assert(re.search(r"etl_\d{8}_\d{6}\.log", LOG_FILE.name), "El log debe incluir fecha y hora")


# EXTRACT devuelve todas las fuentes desde los CSV Bronze.
def test_extract_lee_csv():
    _assert(set(EXTRACTED) == set(FUENTES), "EXTRACT no devolvió todas las fuentes")
    _assert(len(EXTRACTED["erp_ventas"]) == len(BRONZE["erp_ventas"]), "Conteo distinto en erp_ventas")


# Si falta un CSV de fuente, EXTRACT lo regenera y lo persiste.
def test_extract_regenera_si_falta():
    tmp = Path(tempfile.mkdtemp())

    # Regenerador de prueba del test: persiste una fuente de prueba en un directorio dado.
    def _regenerar(ts, directorio):
        guardar_csv(generar_mdm_canales(), "mdm_canales", ts, directorio)

    try:
        datos = extraer("20990101", tmp, ["mdm_canales"], regenerar=_regenerar)
        _assert((tmp / "mdm_canales_20990101.csv").exists(), "No se regeneró el CSV faltante")
        _assert(len(datos["mdm_canales"]) == 3, "mdm_canales regenerado con filas incorrectas")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


# Cada fuente con rechazos tiene su propio archivo; no existe un agregado.
def test_rechazos_por_fuente():
    _assert("crm_clientes" in REJECTED_PATHS, "Falta archivo de rechazos de clientes")
    _assert("erp_ventas" in REJECTED_PATHS, "Falta archivo de rechazos de ventas")
    _assert(REJECTED_PATHS["crm_clientes"] != REJECTED_PATHS["erp_ventas"], "Los rechazos no están separados")
    _assert(len(set(REJECTED_PATHS.values())) == len(REJECTED_PATHS), "Hay rutas de rechazo duplicadas")
    _assert(not list(REJECTED_DIR.glob(f"Rejected_{RUN_DATE_STR}.csv")), "Existe un archivo agregado no permitido")
    for fuente, path in REJECTED_PATHS.items():
        _assert(f"Rejected_{fuente}_" in path.name, f"El archivo no declara la fuente {fuente}")


# El log inicia con INICIO DE PIPELINE y refleja rechazos por fuente.
def test_logs_inicio_y_separacion():
    texto = LOG_FILE.read_text(encoding="utf-8")
    lineas = texto.splitlines()
    _assert(lineas and "INICIO DE PIPELINE" in lineas[0], "El log no inicia con INICIO DE PIPELINE")
    _assert("RECHAZO fuente=crm_clientes" in texto, "El log no registra rechazos de clientes")
    _assert("RECHAZO fuente=erp_ventas" in texto, "El log no registra rechazos de ventas")
    _assert("Rejected_crm_clientes_" in texto and "Rejected_erp_ventas_" in texto,
            "El log no refleja archivos separados de rechazo")


# Los archivos de rechazo contienen la especificación detallada y payload.
def test_rechazo_detalle():
    ruta = REJECTED_PATHS["crm_clientes"]
    df = pd.read_csv(ruta)
    _assert(set(COLUMNAS_RECHAZO) <= set(df.columns), "Faltan columnas de detalle de rechazo")
    _assert(df["motivo_codigo"].notna().all(), "Hay rechazos sin motivo")
    _assert(df["accion_sugerida"].notna().all(), "Hay rechazos sin acción sugerida")
    _assert(df["payload_origen"].notna().all(), "Hay rechazos sin payload original")
    json.loads(df.iloc[0]["payload_origen"])


# Todo motivo emitido existe en MOTIVOS.
def test_motivos_mapeados():
    usados = set()
    for path in REJECTED_PATHS.values():
        usados |= set(pd.read_csv(path)["motivo_codigo"].dropna().unique())
    _assert(usados, "No se registró ningún motivo de rechazo")
    _assert(usados <= set(MOTIVOS), f"Motivos sin mapear: {usados - set(MOTIVOS)}")


# Los RUT inválidos van a rechazos y no a dim_cliente.
def test_transform_rechaza_rut_invalido():
    _assert(len(DIMS_OK["dim_cliente"]) < len(BRONZE["crm_clientes"]), "No se rechazó ningún cliente")
    motivos = {r["motivo_codigo"] for r in RECHAZOS_POR_FUENTE["crm_clientes"]}
    _assert("RUT_INVALIDO" in motivos, "No se documentó el rechazo por RUT inválido")


# Las 10 tablas existen, los conteos coinciden y las FKs están activas.
def test_load_sqlite():
    con = conectar()
    try:
        for nombre in ORDEN_DIMS + ORDEN_FACTS:
            n = con.execute(f'SELECT COUNT(*) FROM "{nombre}"').fetchone()[0]
            _assert(n == CONTEOS[nombre], f"Conteo distinto en {nombre}: {n} != {CONTEOS[nombre]}")
        fk = con.execute("PRAGMA foreign_key_check").fetchall()
        _assert(len(fk) == 0, f"Violaciones de FK: {fk[:3]}")
        _assert(len(con.execute("PRAGMA foreign_key_list('fact_ventas')").fetchall()) >= 4,
                "fact_ventas perdió sus FKs")
    finally:
        con.close()


# La base enmascarada existe, no contiene RUT/email crudos y mantiene hechos.
def test_enmascarado():
    con = conectar(MASKED_DB_NAME, MASKED_DDL)
    try:
        columnas = [r[1] for r in con.execute("PRAGMA table_info('dim_cliente')").fetchall()]
        _assert("rut_enmascarado" in columnas, "Falta rut_enmascarado")
        _assert("email_enmascarado" in columnas, "Falta email_enmascarado")
        _assert("rut_cliente" not in columnas, "La base enmascarada conserva rut_cliente crudo")
        _assert("cliente_id" not in columnas, "La base enmascarada no debe tener una ID de origen")
        muestra = con.execute("SELECT rut_enmascarado, email_enmascarado FROM dim_cliente LIMIT 5").fetchall()
        _assert(muestra, "dim_cliente enmascarada está vacía")
        for rut, email in muestra:
            _assert("***" in str(rut), f"RUT no enmascarado: {rut}")
            _assert("***" in str(email), f"Email no enmascarado: {email}")
        n = con.execute("SELECT COUNT(*) FROM fact_ventas").fetchone()[0]
        _assert(n == CONTEOS["fact_ventas"], "La base enmascarada no conserva fact_ventas")
    finally:
        con.close()


# Con una corrida previa del día, no se crean datos ni se transforma nada.
def test_idempotencia_bloquea_antes_de_generar():
    tmp = Path(tempfile.mkdtemp())
    db = tmp / "idempotencia.db"
    try:
        con = conectar(db)
        con.close()
        registrar_ejecucion("run-previo", 10, 9, 1, run_date="2099-01-01", db_path=db)
        resultado = ejecutar_pipeline(
            run_date="2099-01-01", run_date_str="20990101",
            data_dir=tmp / "data", rejected_dir=tmp / "rejected",
            db_path=db, masked_db_path=tmp / "masked.db",
        )
        _assert(resultado["estado"] == "OMITIDO_POR_IDEMPOTENCIA", "No se omitió la segunda corrida")
        _assert(not (tmp / "data").exists() or not any((tmp / "data").iterdir()),
                "Se generaron datos Bronze pese a la idempotencia")
        _assert(not (tmp / "masked.db").exists(), "Se creó la base enmascarada pese a la idempotencia")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


# Ejecuta todos los tests y reporta PASS/FAIL.
def run_tests():
    pruebas = [v for k, v in sorted(globals().items()) if k.startswith("test_") and callable(v)]
    resultados = []
    for fn in pruebas:
        try:
            fn()
            resultados.append((fn.__name__, "PASS", ""))
            log.info("TEST PASS %s", fn.__name__, extra={"step": "S"})
        except Exception as exc:
            resultados.append((fn.__name__, "FAIL", str(exc)))
            log.error("TEST FAIL %s: %s", fn.__name__, exc, extra={"step": "S"})
    df = pd.DataFrame(resultados, columns=["test", "resultado", "detalle"])
    display(df)
    _assert((df["resultado"] == "PASS").all(), "Hay tests fallidos")
    return df

## 7. Orquestación integral

`ejecutar_pipeline()` verifica idempotencia antes de generar Bronce. Si ya hay una corrida para la fecha,
retorna `OMITIDO_POR_IDEMPOTENCIA` sin crear ni transformar datos.

In [ ]:
# ============================================================
# 7. Orquestación integral
# ============================================================
BRONZE = {}
EXTRACTED = {}
DIMS_OK = {}
FACTS_OK = {}
RECHAZOS_POR_FUENTE = {}
REJECTED_PATHS = {}
CONTEOS = {}
MASKED_CONTEOS = {}
RESULTADO_PIPELINE = None
TESTS = None


# Orquesta EXTRACT → TRANSFORM → LOAD → MASK con idempotencia previa total.
def ejecutar_pipeline(run_date=RUN_DATE, run_date_str=RUN_DATE_STR,
                      data_dir=DATA_DIR, rejected_dir=REJECTED_DIR,
                      db_path=DB_NAME, masked_db_path=MASKED_DB_NAME):
    global BRONZE, EXTRACTED, DIMS_OK, FACTS_OK, RECHAZOS_POR_FUENTE
    global REJECTED_PATHS, CONTEOS, MASKED_CONTEOS, RESULTADO_PIPELINE

    t0 = time.perf_counter()
    previa = ejecucion_previa(run_date, db_path)
    if previa:
        log.warning(
            "IDEMPOTENCIA: pipeline ya ejecutado %s (run_id=%s). "
            "Se omite generación, extracción, transformación, carga y enmascarado.",
            run_date, previa["run_id"], extra={"step": "S"},
        )
        resultado = {"estado": "OMITIDO_POR_IDEMPOTENCIA", "run_date": run_date, "previa": previa}
        RESULTADO_PIPELINE = resultado
        return resultado

    log.info("IDEMPOTENCIA OK: no existe corrida previa para %s", run_date, extra={"step": "S"})

    bronze = generar_bronze(run_date_str, data_dir)
    extraidos = extraer(run_date_str, data_dir)
    rechazos = {fuente: [] for fuente in FUENTES}

    dim_fecha = construir_dim_fecha(extraidos["core_calendario"])
    dim_categoria, categoria_map = construir_dim_categoria(
        extraidos["mdm_categorias"], rechazos["mdm_categorias"], run_date_str)
    dim_producto = construir_dim_producto(
        extraidos["mdm_productos"], categoria_map, rechazos["mdm_productos"], run_date_str)
    dim_sucursal, sucursal_map = construir_dim_sucursal(
        extraidos["mdm_sucursales"], rechazos["mdm_sucursales"], run_date_str)
    dim_cliente, emails, rut_map = construir_dim_cliente(
        extraidos["crm_clientes"], rechazos["crm_clientes"], run_date_str)
    dim_canal, canal_map = construir_dim_canal(
        extraidos["mdm_canales"], rechazos["mdm_canales"], run_date_str)

    dims = {
        "dim_fecha": dim_fecha,
        "dim_categoria": dim_categoria,
        "dim_producto": dim_producto,
        "dim_sucursal": dim_sucursal,
        "dim_cliente": dim_cliente,
        "dim_canal": dim_canal,
    }
    mapas = {
        "producto": dict(zip(dim_producto["sku"], dim_producto["producto_key"])),
        "sucursal": sucursal_map,
        "cliente": rut_map,
        "canal": canal_map,
    }

    facts = {
        "fact_ventas": construir_fact_ventas(
            extraidos["erp_ventas"], dims, mapas, rechazos["erp_ventas"], run_date_str),
        "fact_inventario": construir_fact_inventario(
            extraidos["wms_inventario"], dims, mapas, rechazos["wms_inventario"], run_date_str),
        "fact_campanas": construir_fact_campanas(
            extraidos["mkt_campanas"], dims, mapas, rechazos["mkt_campanas"], run_date_str),
        "fact_devoluciones": construir_fact_devoluciones(
            extraidos["pos_devoluciones"], dims, mapas, rechazos["pos_devoluciones"], run_date_str),
    }

    paths = guardar_rechazos_por_fuente(rechazos, run_date_str, rejected_dir)
    conteos = cargar(dims, facts, db_path)
    masked_conteos = exportar_enmascarado(dims, facts, emails, run_date_str, masked_db_path)

    n_leidas = sum(len(df) for df in extraidos.values())
    n_ok = sum(len(df) for df in list(dims.values()) + list(facts.values()))
    n_rech = len({
        (registro["fuente"], registro["fila_origen"])
        for filas in rechazos.values() for registro in filas
    })
    registrar_ejecucion(RUN_ID, n_leidas, n_ok, n_rech, run_date=run_date, db_path=db_path)

    BRONZE = bronze
    EXTRACTED = extraidos
    DIMS_OK = dims
    FACTS_OK = facts
    RECHAZOS_POR_FUENTE = rechazos
    REJECTED_PATHS = paths
    CONTEOS = conteos
    MASKED_CONTEOS = masked_conteos

    duracion = round(time.perf_counter() - t0, 2)
    log.info(
        "FIN PIPELINE · leídas=%d ok=%d rechazadas=%d · masked=%d tablas · %.2fs",
        n_leidas, n_ok, n_rech, len(masked_conteos), duracion,
        extra={"step": "S"},
    )
    resultado = {
        "estado": "OK",
        "run_date": run_date,
        "run_id": RUN_ID,
        "leidas": n_leidas,
        "ok": n_ok,
        "rechazadas": n_rech,
        "conteos": conteos,
        "masked_conteos": masked_conteos,
        "duracion_s": duracion,
    }
    RESULTADO_PIPELINE = resultado
    return resultado

## 8. Ejecución integral

La primera corrida del día ejecuta todo el flujo y los tests. La segunda corrida solo registra la omisión
por idempotencia.

In [ ]:
resultado = ejecutar_pipeline()
print(json.dumps({k: v for k, v in resultado.items()
                  if k not in ("conteos", "masked_conteos", "previa")},
                 indent=2, ensure_ascii=False))

if resultado.get("estado") == "OK":
    print("\nConteos por tabla (base principal):")
    display(pd.DataFrame([resultado["conteos"]]).T.rename(columns={0: "filas"}))
    print("\nConteos por tabla (base enmascarada):")
    display(pd.DataFrame([resultado["masked_conteos"]]).T.rename(columns={0: "filas"}))
    TESTS = run_tests()
else:
    print("Tests omitidos: el pipeline ya fue ejecutado hoy.")

# Segunda ejecución el mismo día: debe omitirse antes de generar Bronce.
resultado_2 = ejecutar_pipeline()
print("Segunda corrida:", resultado_2["estado"])

# Consulta analítica de ejemplo sobre el datamart.
con = conectar()
try:
    q = """
    SELECT s.region, c.categoria, ROUND(SUM(f.monto_neto), 0) AS venta_neta
    FROM fact_ventas f
    JOIN dim_sucursal s ON s.sucursal_key = f.sucursal_key
    JOIN dim_producto p ON p.producto_key = f.producto_key
    JOIN dim_categoria c ON c.categoria_key = p.categoria_key
    GROUP BY s.region, c.categoria
    ORDER BY venta_neta DESC
    LIMIT 8;
    """
    display(pd.read_sql_query(q, con))
finally:
    con.close()

print("\nArchivos generados:")
for p in sorted(
    list(DATA_DIR.glob(f"*_{RUN_DATE_STR}.csv"))
    + list(REJECTED_DIR.glob(f"*_{RUN_DATE_STR}.csv"))
    + [LOG_FILE, Path(DB_NAME), Path(MASKED_DB_NAME)]
):
    print(" -", p)

## 9. Cierre

- **Bronce:** 10 fuentes heterogéneas con esquemas de origen, RUT válidos ≥ 90% y CSV con `YYYYMMDD`.
- **EXTRACT:** lee los CSV; si falta una fuente, la regenera y persiste.
- **TRANSFORM:** reglas de calidad, rechazo detallado y archivo separado por fuente.
- **LOAD:** `DataRetail360.db` con 6 dimensiones + 4 hechos y FKs activas.
- **MASK:** `DataRetail360_Masked.db` con RUT/email enmascarados y auditoría.
- **Idempotencia:** se verifica antes de crear o transformar cualquier dato.
- **Logging:** `logs/etl_{YYYYMMDD_HHMMSS}.log`, con `INICIO DE PIPELINE` como primera línea.

> Para reiniciar el laboratorio desde cero: elimina `DataRetail360.db`, `DataRetail360_Masked.db`,
> `data/`, `logs/` y `rejected/`, y vuelve a ejecutar todo.